## 1. Setup and installation

Set `INSTALL_PACKAGES = True` on a fresh Colab runtime. Everything a reader might want to
change lives in this section; nothing below hard-codes a threshold or a path.

In [ ]:
# --- run-time switches -------------------------------------------------------
INSTALL_PACKAGES   = True    # True on a fresh Colab runtime
MOUNT_GOOGLE_DRIVE = True    # ignored outside Colab
RUN_SENSITIVITY    = True    # unadjusted Welch t-test alongside the adjusted model
RUN_NETWORKS       = True    # co-expression networks (the slowest section)

# --- path overrides: fill in only if auto-detection fails --------------------
REPO_ROOT_OVERRIDE   = ""
SOURCE_ROOT_OVERRIDE = ""

# --- analysis parameters -----------------------------------------------------
CONFIG = {
    "dataset": "GSE69683",
    # GSE69683 was run on GPL13158, the Affymetrix HT HG-U133+ PM Array Plate
    # (54,715 probes). This is a different design from GPL570 / HG-U133 Plus 2.0
    # (54,675 probes), although the two share probe set identifiers.
    "platform": "GPL13158",
    "expected_samples": 24,
    "expected_probes": 54715,
    "expected_ancestry_counts": {"African": 12, "European": 12},
    "expected_sex_counts": {"Female": 16, "Male": 8},

    "fdr_threshold": 0.05,
    "absolute_log2fc_threshold": 0.5,

    "random_seed": 42,
    "tsne_perplexity": 8,

    "gsea_min_size": 5,
    "gsea_max_size": 500,
    "gsea_permutations": 1000,

    "n_top_de_genes": 50,     # heatmap size
    "n_volcano_labels": 15,   # annotated genes per volcano

    "coexpression_abs_r": 0.60,
    "coexpression_fdr": 0.05,
}

RANDOM_SEED = CONFIG["random_seed"]

# Kept so the notebook can assert that its inputs match the intended design.
# These are properties of the study, not results.
EXPECTED = {
    "series_samples": 498,
    "african_race_value": "race: black_african",
    "european_race_value": "race: white_caucasian",
}

In [ ]:
REQUIRED_PACKAGES = [
    "numpy>=1.26", "pandas>=2.2", "scipy>=1.13", "statsmodels>=0.14",
    "scikit-learn>=1.4", "matplotlib>=3.8", "seaborn>=0.13",
    "GEOparse>=2.0", "gseapy>=1.1", "networkx>=3.2", "requests>=2.31",
]

if INSTALL_PACKAGES:
    import subprocess, sys
    command = [sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0 and "externally-managed-environment" in result.stderr:
        # Some system Pythons refuse to install without this flag.
        result = subprocess.run(command + ["--break-system-packages"],
                                capture_output=True, text=True)
    if result.returncode == 0:
        print("Packages installed.")
    else:
        # Do not abort the run: the packages may already be present.
        print("pip returned a non-zero exit code. If the imports in the next cell all")
        print("succeed you can ignore this. Otherwise install manually. pip said:")
        print((result.stderr or result.stdout).strip()[-600:])
else:
    print("INSTALL_PACKAGES is False. Set it True if the imports below fail.")
    print("On a fresh Colab runtime you almost certainly need it True.")

## 2. Import libraries

In [ ]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import os
import platform
import re
import shutil
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
np.random.seed(RANDOM_SEED)

RUN_STARTED = datetime.now(timezone.utc).isoformat(timespec="seconds")

# Shared palettes, so every figure uses the same colours.
PALETTE_ANCESTRY = {"African": "#C65D4B", "European": "#4C78A8"}
PALETTE_GENDER   = {"Female": "#7A5195", "Male": "#2A9D8F"}
MARKERS_GENDER   = {"Female": "^", "Male": "o"}
COLOUR_UP        = "#C0392B"
COLOUR_DOWN      = "#2471A3"
COLOUR_NS        = "#BDBDBD"

print("Python           :", sys.version.split()[0])
print("Platform         :", platform.platform())
print("Run started (UTC):", RUN_STARTED)
print("Random seed      :", RANDOM_SEED)

## 3. Paths and Google Drive

Outputs are written under `results/manuscript_1/` and never mixed with inputs.

In [ ]:
# --- where the project lives -------------------------------------------------
# On Colab, outputs MUST go to Drive. /content is the runtime's temporary disk
# and is wiped when the session disconnects, so anything written there is lost
# and cannot be shared as a link.
DRIVE_PROJECT_NAME = "cross-ancestry-asthma-transcriptomics"

IN_COLAB = "google.colab" in sys.modules
DRIVE_MOUNTED = False

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = Path("/content/drive/MyDrive").is_dir()


def first_existing(candidates):
    for candidate in candidates:
        path = Path(candidate).expanduser()
        if path.exists() and path.stat().st_size > 0:
            return path
    return None


def looks_like_repo_root(path: Path) -> bool:
    return path.exists() and (path / "data").is_dir() and (path / "notebooks").is_dir()


def find_repo_root(start: Path):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if looks_like_repo_root(candidate):
            return candidate
    return None


# Resolution order:
#   1. an explicit override, if set
#   2. an existing project folder at or above the working directory (local clone)
#   3. on Colab with Drive mounted: a project folder in Drive, created if absent
#   4. the working directory (local) or /content (Colab, with a warning)
if REPO_ROOT_OVERRIDE:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).expanduser()
    root_source = "REPO_ROOT_OVERRIDE"
elif (found := find_repo_root(Path.cwd())) is not None:
    REPO_ROOT = found
    root_source = "existing project folder near the working directory"
elif DRIVE_MOUNTED:
    REPO_ROOT = Path("/content/drive/MyDrive") / DRIVE_PROJECT_NAME
    root_source = "Google Drive (created if it did not exist)"
else:
    REPO_ROOT = Path("/content") if IN_COLAB else Path.cwd()
    root_source = "fallback"

REPO_ROOT.mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "notebooks").mkdir(exist_ok=True)   # so this folder is found next time

# Is this location going to survive the session?
PERSISTENT = (not IN_COLAB) or str(REPO_ROOT).startswith("/content/drive/")

if SOURCE_ROOT_OVERRIDE:
    SOURCE_ROOT = Path(SOURCE_ROOT_OVERRIDE).expanduser()
else:
    SOURCE_ROOT = next(
        (c for c in [Path("/content/drive/MyDrive/Asthma_Transcriptomics_Results"), REPO_ROOT]
         if c.exists()), REPO_ROOT)

DATA_RAW        = REPO_ROOT / "data" / "raw"
DATA_PROCESSED  = REPO_ROOT / "data" / "processed"
DATA_ANNOTATION = REPO_ROOT / "data" / "annotation"
DATA_GENE_SETS  = REPO_ROOT / "data" / "gene_sets"
DATA_MANIFESTS  = REPO_ROOT / "data" / "manifests"

MS1          = REPO_ROOT / "results" / "manuscript_1"
MAIN_TABLES  = MS1 / "main_tables"
MAIN_FIGURES = MS1 / "main_figures"
SUPPLEMENT   = MS1 / "supplementary"
S1_DIR = SUPPLEMENT / "S1_complete_DE_and_expression"
S2_DIR = SUPPLEMENT / "S2_pathway_and_enrichment"
S3_DIR = SUPPLEMENT / "S3_curated_gene_statistics"
S4_DIR = SUPPLEMENT / "S4_coexpression_networks"
FIG_PATHWAY  = MAIN_FIGURES / "pathway_violins"

for directory in [DATA_RAW, DATA_PROCESSED, DATA_ANNOTATION, DATA_GENE_SETS, DATA_MANIFESTS,
                  MAIN_TABLES, MAIN_FIGURES, FIG_PATHWAY, S1_DIR, S2_DIR, S3_DIR, S4_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def save_figure(figure, path, *, dpi=300):
    """Every figure in this notebook is saved through here, at a single DPI."""
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=dpi, bbox_inches="tight")
    try:
        print("Saved figure:", path.relative_to(REPO_ROOT))
    except ValueError:
        print("Saved figure:", path)


print("Project root :", REPO_ROOT)
print("Resolved from:", root_source)
print("Search path  :", SOURCE_ROOT)
print("Output root  :", MS1)
print()
if PERSISTENT:
    print("Storage is PERSISTENT. Outputs will survive this session and the folder")
    print("can be shared from Drive.")
else:
    print("=" * 70)
    print("WARNING: outputs are going to TEMPORARY storage and will be LOST when")
    print("this runtime disconnects. They also cannot be shared as a link.")
    print("=" * 70)
    if IN_COLAB:
        print("Fix: set MOUNT_GOOGLE_DRIVE = True in section 1, or set")
        print("REPO_ROOT_OVERRIDE to a path under /content/drive/MyDrive/, then")
        print("re-run from the top.")


## 4. Data source

Expression data come from **GSE69683**, the whole-blood transcriptomic arm of the
U-BIOPRED severe asthma cohort (Bigler et al., 2017). The full series contains 498
samples profiled on the Affymetrix HT HG-U133+ PM Array Plate (GPL13158), 54,715 probe
sets, supplied RMA-normalised and log2-transformed. The data are openly available from
NCBI GEO without restriction.

The notebook downloads the series matrix directly rather than reading a prepared file, so
the input is the same for anyone who runs it, and records the SHA-256 of the downloaded
file so the exact input can be verified later.

The download is roughly 250 MB and takes a few minutes on a Colab runtime. If the file is
already present in `data/raw/` it is reused.

In [ ]:
SERIES_MATRIX_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE69nnn/GSE69683/matrix/"
    "GSE69683_series_matrix.txt.gz"
)

series_matrix_path = first_existing([
    DATA_RAW / "GSE69683_series_matrix.txt.gz",
    SOURCE_ROOT / "GSE69683_series_matrix.txt.gz",
    Path.cwd() / "GSE69683_series_matrix.txt.gz",
])

if series_matrix_path is None:
    import requests
    series_matrix_path = DATA_RAW / "GSE69683_series_matrix.txt.gz"
    print("Downloading the series matrix from GEO. This takes a few minutes.")
    with requests.get(SERIES_MATRIX_URL, stream=True, timeout=900) as response:
        response.raise_for_status()
        with series_matrix_path.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1 << 20):
                if chunk:
                    handle.write(chunk)
    print("Downloaded:", series_matrix_path)


def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(block_size):
            digest.update(chunk)
    return digest.hexdigest()


SERIES_MATRIX_SHA256 = sha256_file(series_matrix_path)
print("Series matrix :", series_matrix_path)
print("Size (MB)     :", round(series_matrix_path.stat().st_size / 1e6, 1))
print("SHA-256       :", SERIES_MATRIX_SHA256)

In [ ]:
def read_series_matrix_header(path: Path) -> dict[str, list[list[str]]]:
    """Every '!Sample_*' header line, keyed by field name. Characteristics repeat,
    so values accumulate in a list and the caller picks the right row."""
    header: dict[str, list[list[str]]] = {}
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if line.startswith("!series_matrix_table_begin"):
                break
            if not line.startswith("!Sample_"):
                continue
            key, _, rest = line.partition("\t")
            values = [v.strip().strip('"') for v in rest.rstrip("\n").split("\t")]
            header.setdefault(key.strip(), []).append(values)
    return header


def read_series_matrix_table(path: Path) -> pd.DataFrame:
    """The expression block as probes x samples, all numeric."""
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        lines = handle.readlines()
    begin = next(i for i, l in enumerate(lines) if "!series_matrix_table_begin" in l) + 1
    end = next(i for i, l in enumerate(lines) if "!series_matrix_table_end" in l)
    frame = pd.read_csv(io.StringIO("".join(lines[begin:end])), sep="\t", index_col=0)
    frame.index = frame.index.astype(str).str.strip().str.strip('"')
    frame.columns = frame.columns.astype(str).str.strip().str.strip('"')
    frame = frame.apply(pd.to_numeric, errors="coerce")
    frame.index.name = "probe_id"
    return frame


header = read_series_matrix_header(series_matrix_path)
sample_ids = header["!Sample_geo_accession"][0]
characteristics = header.get("!Sample_characteristics_ch1", [])

print("Samples in the full series :", len(sample_ids))
print("Characteristics rows       :", len(characteristics))
for i, row in enumerate(characteristics):
    print(f"  [{i}] {len(set(row)):>3} distinct | example: {row[0][:70] if row else ''}")

## 5. Sample metadata and cohort definition

Ancestry and sex are parsed from the GEO series matrix characteristics fields rather than
assigned by column position, so the labels can be traced to the public record.

The 24-sample cohort is defined by a manifest committed to the repository. This is
necessary because GSE69683 contains 498 samples and nothing in the GEO record identifies
which subset a given study analysed: the selection is a design decision that cannot be
re-derived computationally. Where the manifest carries ancestry or sex labels they are
cross-checked against GEO, and the notebook halts on any disagreement.

In [ ]:
ANCESTRY_PATTERNS = {
    "African":  re.compile(r"(afric|\bblack\b)", re.I),
    "European": re.compile(r"(europ|caucas|\bwhite\b)", re.I),
}
GENDER_PATTERNS = {
    "Female": re.compile(r"\bfemale\b", re.I),
    "Male":   re.compile(r"(?<!fe)\bmale\b", re.I),
}


def label_from_characteristics(values, patterns):
    labels = []
    for value in values:
        matched = [name for name, pattern in patterns.items() if pattern.search(str(value))]
        labels.append(matched[0] if len(matched) == 1 else None)
    return labels


def choose_characteristics_row(rows, patterns):
    best, best_score, best_index = None, 0, None
    for index, row in enumerate(rows):
        labels = label_from_characteristics(row, patterns)
        score = sum(l is not None for l in labels)
        if score > best_score:
            best, best_score, best_index = labels, score, index
    return best, best_score, best_index


ancestry_labels, ancestry_score, ancestry_row = choose_characteristics_row(
    characteristics, ANCESTRY_PATTERNS)
gender_labels, gender_score, gender_row = choose_characteristics_row(
    characteristics, GENDER_PATTERNS)

derived = pd.DataFrame({
    "sample_id": sample_ids,
    "ancestry": ancestry_labels if ancestry_labels else [None] * len(sample_ids),
    "gender": gender_labels if gender_labels else [None] * len(sample_ids),
    "geo_race_value": (characteristics[ancestry_row] if ancestry_row is not None
                       else [None] * len(sample_ids)),
})

print(f"Ancestry labelled from characteristics row {ancestry_row}: "
      f"{ancestry_score}/{len(sample_ids)}")
print(f"Gender   labelled from characteristics row {gender_row}: "
      f"{gender_score}/{len(sample_ids)}")
print("\nFull-series composition, before cohort selection:")
print(derived[["ancestry", "gender"]].value_counts(dropna=False).to_string())

In [ ]:
# --- cohort manifest ---------------------------------------------------------
# The manifest is the list of which samples were analysed. GSE69683 has hundreds
# of samples; this study uses 24. Nothing in the GEO record says which 24, so the
# list must be version-controlled rather than living in a spreadsheet.

MANIFEST_PATH = DATA_MANIFESTS / "sample_manifest.tsv"

MS1_COHORT = {
    "African":  ["GSM1706863", "GSM1707019", "GSM1707032", "GSM1707059",
                 "GSM1707104", "GSM1707162", "GSM1707174", "GSM1707227",
                 "GSM1707234", "GSM1707252", "GSM1707271", "GSM1707329"],
    "European": ["GSM1706889", "GSM1706913", "GSM1706933", "GSM1706935",
                 "GSM1706941", "GSM1706948", "GSM1706996", "GSM1707014",
                 "GSM1707022", "GSM1707028", "GSM1707082", "GSM1707122"],
}
MS1_MALES = {"GSM1707032", "GSM1707059", "GSM1707227", "GSM1707234",
             "GSM1706941", "GSM1707028", "GSM1707082", "GSM1707122"}

if not (MANIFEST_PATH.exists() and MANIFEST_PATH.stat().st_size > 0):
    rows = [
        {"sample_id": s, "included": True, "ancestry": ancestry,
         "gender": "Male" if s in MS1_MALES else "Female",
         "source": "GSE69683",
         "notes": "MS1 cohort, recovered from the exploratory notebook output"}
        for ancestry, samples in MS1_COHORT.items() for s in samples
    ]
    pd.DataFrame(rows).to_csv(MANIFEST_PATH, sep="\t", index=False)
    print("Wrote the cohort manifest:", MANIFEST_PATH)

manifest = pd.read_csv(MANIFEST_PATH, sep="\t", comment="#")
manifest["sample_id"] = manifest["sample_id"].astype(str).str.strip()
if "included" in manifest.columns:
    keep = manifest["included"].astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})
    manifest = manifest.loc[keep]

selected_ids = manifest["sample_id"].drop_duplicates().tolist()
manifest_source = f"committed manifest: {MANIFEST_PATH}"
print("Cohort source   :", manifest_source)
print("Samples selected:", len(selected_ids))

### 5a. How the ancestry groups are defined

GSE69683 records a `race` field with nine distinct values across the full series. This
section prints them, and prints the value GEO holds for each of the 24 cohort samples, so
the ancestry definition used here is explicit and checkable rather than asserted.

The two groups correspond to single, unambiguous GEO values: `race: black_african` and
`race: white_caucasian`. No categories are pooled.

**This is self-reported race or ethnicity as recorded by U-BIOPRED, not genetically
inferred ancestry.** No genotype data are available for these samples, so genetic ancestry
cannot be estimated. Self-reported race is a coarse and imperfect proxy for genetic
ancestry, and results should be read with that limitation in mind throughout.

In [ ]:
race_series = pd.Series(characteristics[ancestry_row], index=sample_ids, name="geo_race_value")

print("ALL DISTINCT VALUES IN THE GEO RACE FIELD (full series)")
print("=" * 72)
print(race_series.value_counts(dropna=False).to_string())

cohort_race = pd.DataFrame({
    "geo_race_value": race_series.reindex(selected_ids),
    "manifest_label": (manifest.set_index("sample_id")["ancestry"].reindex(selected_ids)
                       if "ancestry" in manifest.columns else "(none in manifest)"),
})

print("\nWHAT GEO SAYS ABOUT THE 24 COHORT SAMPLES")
print("=" * 72)
print(cohort_race.to_string())

if "ancestry" in manifest.columns:
    print("\nManifest label versus GEO race value")
    print("=" * 72)
    print(pd.crosstab(cohort_race["manifest_label"], cohort_race["geo_race_value"]).to_string())

cohort_race.to_csv(DATA_MANIFESTS / "MS1_ancestry_label_provenance.tsv", sep="\t")
print("\nSaved:", (DATA_MANIFESTS / "MS1_ancestry_label_provenance.tsv").name)
print()
print("Report this as self-reported race or ethnicity as recorded by U-BIOPRED,")
print("not genetically inferred ancestry. No genotype data exist for these samples,")
print("and self-reported race is an imperfect proxy for genetic ancestry.")

In [ ]:
metadata = derived.set_index("sample_id").reindex(selected_ids).copy()

# Fall back to manifest labels only where GEO could not label a sample.
for column in ["ancestry", "gender"]:
    if column in manifest.columns:
        supplied = (manifest.set_index("sample_id")[column]
                    .astype(str).str.strip().str.title().reindex(metadata.index))
        metadata[column] = metadata[column].fillna(supplied)

# Where both sources have a label they must agree.
conflicts = []
for column in ["ancestry", "gender"]:
    if column in manifest.columns:
        supplied = (manifest.set_index("sample_id")[column]
                    .astype(str).str.strip().str.title().reindex(metadata.index))
        geo = derived.set_index("sample_id")[column].reindex(metadata.index)
        clash = metadata.index[supplied.notna() & geo.notna() & (supplied != geo)]
        for sample in clash:
            conflicts.append({"sample_id": sample, "field": column,
                              "from_GEO": geo[sample], "from_manifest": supplied[sample]})

if conflicts:
    print("METADATA CONFLICT: the manifest disagrees with the GEO series matrix.")
    print(pd.DataFrame(conflicts).to_string(index=False))
    raise ValueError("Resolve the metadata conflicts above before analysing.")

metadata = metadata[["ancestry", "gender", "geo_race_value"]]
metadata.index.name = "sample_id"

missing = metadata.index[metadata["ancestry"].isna() | metadata["gender"].isna()]
if len(missing):
    raise ValueError(f"Ancestry or gender unresolved for: {list(missing)}")

print("Cohort composition")
print("  ancestry:", metadata["ancestry"].value_counts().to_dict())
print("  gender  :", metadata["gender"].value_counts().to_dict())
print("\nAncestry x gender:")
print(pd.crosstab(metadata["ancestry"], metadata["gender"]).to_string())

assert len(metadata) == CONFIG["expected_samples"]
assert metadata["ancestry"].value_counts().to_dict() == CONFIG["expected_ancestry_counts"]
assert metadata["gender"].value_counts().to_dict() == CONFIG["expected_sex_counts"]

print()
print("Note the design: each ancestry group is 8 female and 4 male. Sex is balanced")
print("ACROSS the ancestry groups, which is what makes the adjusted model identifiable,")
print("but the cohort is not balanced WITHIN each group.")

metadata.reset_index().to_csv(DATA_PROCESSED / "GSE69683_24sample_metadata.tsv",
                              sep="\t", index=False)

In [ ]:
# Selection audit: how large was the pool the cohort was drawn from?
labelled = derived.dropna(subset=["ancestry", "gender"])
pool = pd.crosstab(labelled["ancestry"], labelled["gender"])
chosen = pd.crosstab(metadata["ancestry"], metadata["gender"])

rows = []
for ancestry in chosen.index:
    for gender in chosen.columns:
        available = int(pool.loc[ancestry, gender]) if (
            ancestry in pool.index and gender in pool.columns) else 0
        taken = int(chosen.loc[ancestry, gender])
        rows.append({"ancestry": ancestry, "gender": gender,
                     "available_in_series": available, "selected": taken,
                     "selection_fraction": round(taken / available, 4) if available else None})

selection_audit = pd.DataFrame(rows)
print("Samples in the series ....", len(derived))
print("Selected for this study ..", len(metadata))
print()
print(selection_audit.to_string(index=False))
print()
print("Any report using this cohort should state HOW the samples were chosen from the")
print("available pool. If matched on age, severity or treatment, say so; a subset with")
print("no stated criterion invites the question of post-hoc selection.")

selection_audit.to_csv(DATA_MANIFESTS / "MS1_cohort_selection_audit.tsv", sep="\t", index=False)

(DATA_PROCESSED / "MS1_cohort_provenance.json").write_text(json.dumps({
    "run_started_utc": RUN_STARTED,
    "series_accession": CONFIG["dataset"],
    "samples_in_series": int(len(derived)),
    "samples_selected": int(len(metadata)),
    "cohort_source": manifest_source,
    "series_matrix_sha256": SERIES_MATRIX_SHA256,
    "sample_ids": metadata.index.tolist(),
}, indent=2), encoding="utf-8")

In [ ]:
# --- expression matrix and platform verification -----------------------------
expression_all = read_series_matrix_table(series_matrix_path)

absent = sorted(set(metadata.index) - set(expression_all.columns))
if absent:
    raise ValueError(f"In the manifest but absent from the expression matrix: {absent}")

expression = expression_all.loc[:, metadata.index].copy()
assert list(expression.columns) == list(metadata.index)

n_probes = expression.shape[0]
pm_probes = int(expression.index.str.contains("_PM_", regex=False).sum())

print("Full series matrix    :", expression_all.shape)
print("Selected cohort matrix:", expression.shape)
print(f"Probes carrying '_PM_': {pm_probes:,} of {n_probes:,}")

detected = ("GPL13158 (HT HG-U133+ PM Array Plate)"
            if n_probes == 54715 or pm_probes > 0.5 * n_probes
            else "GPL570 (HG-U133 Plus 2.0)" if n_probes == 54675
            else f"unrecognised ({n_probes} probes)")

print("\nDetected platform:", detected)
if CONFIG["platform"] not in detected:
    raise ValueError(
        f"Expected {CONFIG['platform']} but the matrix looks like {detected}. "
        "Check the series matrix before continuing."
    )
print("The _PM_ infix is specific to the HT HG-U133+ PM design and is what")
print("distinguishes GPL13158 from GPL570, which carries 54,675 probes.")

assert n_probes == CONFIG["expected_probes"]
expression.to_csv(DATA_PROCESSED / "GSE69683_24sample_probe_expression.csv.gz", compression="gzip")

## 6. Distribution study

Three panels, as in the original notebook: sample-wise boxplots, then expression density
split by ancestry and by gender.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(20, 5))

expression.plot(kind="box", ax=axes[0], showfliers=True,
                color=dict(boxes="black", whiskers="black", medians="black", caps="black"))
axes[0].set_title("Sample-wise Expression Distributions", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Samples")
axes[0].set_ylabel("Log$_2$ expression")
axes[0].tick_params(axis="x", rotation=90, labelsize=6)

for panel, (column, palette) in zip(axes[1:], [("ancestry", PALETTE_ANCESTRY),
                                               ("gender", PALETTE_GENDER)]):
    for level, colour in palette.items():
        ids = metadata.index[metadata[column] == level]
        panel.hist(expression[ids].to_numpy().ravel(), bins=60, density=True,
                   alpha=0.45, label=level, color=colour)
    panel.set_title(f"Expression Distribution by {column.title()}",
                    fontsize=12, fontweight="bold")
    panel.set_xlabel("Log$_2$ expression")
    panel.set_ylabel("Density")
    panel.legend()

figure.tight_layout()
save_figure(figure, MAIN_FIGURES / "01_sample_distributions_with_gender.png")
plt.show()

## 7. Quality control

Distributional summaries and pairwise sample correlations. These identify gross anomalies
and outlier samples. They are not a test for batch effects: GSE69683 does not expose array
plate or processing batch, so no batch covariate is available.

In [ ]:
qc_summary = pd.Series({
    "probe_count": expression.shape[0],
    "sample_count": expression.shape[1],
    "missing_values": int(expression.isna().sum().sum()),
    "minimum": float(expression.min().min()),
    "maximum": float(expression.max().max()),
    "mean": float(expression.to_numpy().mean()),
    "median": float(np.median(expression.to_numpy())),
}, name="value")
print(qc_summary.to_frame().to_string())

if qc_summary["missing_values"] != 0:
    raise ValueError("Missing expression values detected. Resolve before analysis.")

sample_correlations = expression.corr(method="pearson")
upper = sample_correlations.where(np.triu(np.ones(sample_correlations.shape), k=1).astype(bool))
pairwise = upper.stack()

print(f"\nPairwise Pearson correlation: mean {pairwise.mean():.4f}, "
      f"range {pairwise.min():.4f} to {pairwise.max():.4f}")
print("\nHigh, tightly-clustered correlations indicate consistent technical quality.")
print("This identifies outlier samples; it is not a test for batch effects, and no")
print("batch covariate is available in the GEO metadata.")

qc_summary.to_csv(MAIN_TABLES / "MS1_QC_summary.tsv", sep="\t")
sample_correlations.to_csv(S1_DIR / "MS1_S1_sample_correlation_matrix.tsv", sep="\t")

## 8. Exploratory data analysis: PCA and t-SNE

Ancestry is shown by colour and sex by marker shape.

t-SNE parameters are stated explicitly (perplexity 8, PCA initialisation, seed 42). On 24
samples t-SNE will separate almost any grouping, so it is included as a visualisation
rather than as evidence of structure; PCA carries the interpretable variance.

In [ ]:
sample_by_probe = expression.T

pca = PCA(n_components=min(10, sample_by_probe.shape[0] - 1), random_state=RANDOM_SEED)
pca_coordinates = pca.fit_transform(sample_by_probe)
variance = pca.explained_variance_ratio_ * 100

pca_df = pd.DataFrame(pca_coordinates[:, :3], index=sample_by_probe.index,
                      columns=["PC1", "PC2", "PC3"]).join(metadata[["ancestry", "gender"]])

print("Variance explained")
for i in range(3):
    print(f"  PC{i + 1}: {variance[i]:.2f}%")
print(f"  PC1-3 cumulative: {variance[:3].sum():.2f}%")

tsne = TSNE(n_components=2, perplexity=CONFIG["tsne_perplexity"], init="pca",
            learning_rate="auto", random_state=RANDOM_SEED)
tsne_coordinates = tsne.fit_transform(pca_coordinates)
tsne_df = pd.DataFrame(tsne_coordinates, index=sample_by_probe.index,
                       columns=["tSNE1", "tSNE2"]).join(metadata[["ancestry", "gender"]])

pca_df.to_csv(S1_DIR / "MS1_S1_PCA_coordinates.tsv", sep="\t")
tsne_df.to_csv(S1_DIR / "MS1_S1_tSNE_coordinates.tsv", sep="\t")

In [ ]:
# Adjusted PCA figure: ancestry by colour, gender by shape.
figure, axes = plt.subplots(1, 2, figsize=(16, 6))

for ancestry, colour in PALETTE_ANCESTRY.items():
    for gender, marker in MARKERS_GENDER.items():
        mask = (pca_df["ancestry"] == ancestry) & (pca_df["gender"] == gender)
        axes[0].scatter(pca_df.loc[mask, "PC1"], pca_df.loc[mask, "PC2"],
                        label=f"{ancestry}, {gender}", color=colour, marker=marker,
                        s=140, edgecolor="black", linewidth=0.6, alpha=0.9)
axes[0].set_title("PCA: PC1 vs PC2 by Ancestry and Gender", fontsize=15, fontweight="bold")
axes[0].set_xlabel(f"PC1 ({variance[0]:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({variance[1]:.1f}% variance)")
axes[0].axhline(0, color="grey", linewidth=0.6, linestyle=":")
axes[0].axvline(0, color="grey", linewidth=0.6, linestyle=":")
axes[0].legend(fontsize=9)

axes[1].bar(np.arange(1, len(variance) + 1), variance, color="#4C78A8", alpha=0.85)
axes[1].plot(np.arange(1, len(variance) + 1), np.cumsum(variance),
             marker="o", color="#C65D4B", label="Cumulative")
axes[1].set_title("Scree Plot: Variance Explained", fontsize=15, fontweight="bold")
axes[1].set_xlabel("Principal component")
axes[1].set_ylabel("Variance explained (%)")
axes[1].legend(fontsize=9)

figure.tight_layout()
save_figure(figure, MAIN_FIGURES / "02_pca_analysis.png")
plt.show()

In [ ]:
figure, axis = plt.subplots(figsize=(8, 6))
for ancestry, colour in PALETTE_ANCESTRY.items():
    for gender, marker in MARKERS_GENDER.items():
        mask = (tsne_df["ancestry"] == ancestry) & (tsne_df["gender"] == gender)
        axis.scatter(tsne_df.loc[mask, "tSNE1"], tsne_df.loc[mask, "tSNE2"],
                     label=f"{ancestry}, {gender}", color=colour, marker=marker,
                     s=140, edgecolor="black", linewidth=0.6, alpha=0.9)
axis.set_title(f"t-SNE (perplexity={CONFIG['tsne_perplexity']}, seed={RANDOM_SEED})",
               fontsize=15, fontweight="bold")
axis.set_xlabel("t-SNE 1")
axis.set_ylabel("t-SNE 2")
axis.legend(fontsize=9)
figure.tight_layout()
save_figure(figure, MAIN_FIGURES / "03_tsne_by_ancestry_gender.png")
plt.show()

print("t-SNE on 24 samples separates groups readily and is not evidence of structure")
print("beyond what PCA already shows. Present it as a visualisation, not as confirmation.")

## 9. Hierarchical clustering and correlation matrix

Two panels: the dendrogram and the clustered correlation matrix, as in the original.

In [ ]:
# NOTE: .to_numpy(copy=True) matters. pandas >= 2 can return a read-only view, and
# np.fill_diagonal writes in place, so calling it on .values raises
# "ValueError: underlying array is read-only". The exploratory notebook has this bug.
correlation_distance = (1 - sample_correlations).clip(lower=0).to_numpy(copy=True)
np.fill_diagonal(correlation_distance, 0.0)
correlation_distance = (correlation_distance + correlation_distance.T) / 2
row_linkage = linkage(squareform(correlation_distance, checks=False), method="average")

figure, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

label_colours = [PALETTE_ANCESTRY[metadata.loc[s, "ancestry"]] for s in sample_correlations.index]
dendrogram(row_linkage, labels=sample_correlations.index.tolist(), ax=ax1,
           leaf_rotation=90, leaf_font_size=8, color_threshold=0)
ax1.set_title("Hierarchical Clustering Dendrogram", fontsize=14, fontweight="bold")
ax1.set_ylabel("Distance (1 - Pearson r)")
for tick, sample in zip(ax1.get_xticklabels(), [t.get_text() for t in ax1.get_xticklabels()]):
    if sample in metadata.index:
        tick.set_color(PALETTE_ANCESTRY[metadata.loc[sample, "ancestry"]])

order = [sample_correlations.index[i] for i in
         dendrogram(row_linkage, no_plot=True)["leaves"]]
sns.heatmap(sample_correlations.loc[order, order], cmap="vlag",
            center=float(sample_correlations.to_numpy().mean()),
            square=True, ax=ax2, cbar_kws={"label": "Pearson r"},
            xticklabels=True, yticklabels=True)
ax2.set_title("Sample Correlation Matrix (Clustered)", fontsize=14, fontweight="bold")
ax2.tick_params(labelsize=7)

handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10, label=a)
           for a, c in PALETTE_ANCESTRY.items()]
ax1.legend(handles=handles, title="Ancestry", fontsize=9)

figure.tight_layout()
save_figure(figure, MAIN_FIGURES / "04_hierarchical_clustering.png")
plt.show()

In [ ]:
# Annotated clustermap, the second clustering view from the original notebook.
column_colours = pd.DataFrame({
    "Ancestry": metadata["ancestry"].map(PALETTE_ANCESTRY),
    "Gender": metadata["gender"].map(PALETTE_GENDER),
}, index=metadata.index)

grid = sns.clustermap(sample_correlations, row_linkage=row_linkage, col_linkage=row_linkage,
                      cmap="vlag", center=float(sample_correlations.to_numpy().mean()),
                      col_colors=column_colours, figsize=(11, 10),
                      xticklabels=True, yticklabels=True,
                      cbar_kws={"label": "Pearson r"})
grid.fig.suptitle("Clustered Sample-Correlation Matrix with Annotations", y=1.02,
                  fontsize=14, fontweight="bold")
save_figure(grid.fig, MAIN_FIGURES / "05_sample_correlation_annotated.png")
plt.show()

### 9b. Curated pathway definitions

One controlled source. The TSV is authoritative; the GMT is generated from it. This
replaces the multiple, partly inconsistent GMT files built in the exploratory notebook.

In [ ]:
PATHWAY_TSV = DATA_GENE_SETS / "asthma_curated_pathways.tsv"
PATHWAY_GMT = DATA_GENE_SETS / "asthma_curated_pathways.gmt"

FALLBACK_PATHWAYS = {
    "Immune_Response": ["IL4", "IL5", "IL13", "IL33", "TSLP", "IL4R", "IL5RA", "IL13RA1",
                        "CD28", "CTLA4", "ICOS", "GATA3", "TBX21", "STAT6", "STAT4",
                        "CCL11", "CCL24", "CCL26", "CCR3", "TNFSF4", "IL1RL1"],
    "Airway_Remodeling": ["TGFB1", "TGFB2", "TGFBR1", "TGFBR2", "SMAD2", "SMAD3", "SMAD4",
                          "COL1A1", "COL3A1", "FN1", "MMP9", "MMP12", "TIMP1", "TIMP2",
                          "ACTA2", "PDGFA", "PDGFB", "CTGF", "EDN1", "FGF2"],
    "Oxidative_Stress": ["NOS2", "NOX1", "NOX4", "SOD2", "CAT", "GPX1", "GPX3", "HMOX1",
                         "NQO1", "NFE2L2", "KEAP1", "TXNRD1", "PRDX1", "GCLC", "GCLM",
                         "GSR", "GSTP1"],
    "Bronchial_Hyperresponsiveness": ["ADRB2", "CHRM3", "EDN1", "EDNRA", "EDNRB", "PTGER2",
                                      "PTGER4", "LTC4S", "ALOX5", "CYSLTR1", "CYSLTR2",
                                      "VIP", "VIPR1", "ADCY1", "PDE4D", "KCNMA1"],
    "Epithelial_Barrier_Function": ["MUC5AC", "MUC5B", "CLDN1", "CLDN18", "TJP1", "SPDEF",
                                    "FOXJ1", "FLG", "TP63"],
    "Innate_Immunity_TLR": ["TLR2", "TLR4", "MYD88", "NFKB1", "RELA", "IL1B", "CXCL8",
                            "CXCL10", "CCL20"],
    "Type17_Neutrophilic": ["IL17A", "IL17F", "IL22", "RORC", "CXCL1", "CXCL5", "CXCL10",
                            "CCL20"],
    "Hypoxia_Metabolic_Response": ["HIF1A", "VEGFA", "LDHA", "PPARGC1A", "SLC2A1"],
    "Steroid_Response": ["NR3C1", "FKBP5", "TSC22D3", "DUSP1", "ZBTB16"],
    "Drug_Metabolism": ["NR3C1", "FKBP5", "HSD11B1", "HSD11B2", "TSC22D3", "DUSP1", "ADRB2",
                        "PDE4D", "ARRB2", "GSTM1", "GSTM3", "GSTA1", "GSTA2", "GSTP1",
                        "CYP3A4", "CYP3A5", "CYP2D6", "CYP2C9", "CYP2C19", "CYP1A1",
                        "CYP1B1", "UGT1A1", "NAT2", "SULT1A1", "ABCB1", "ABCC1", "SLCO1B1",
                        "ABCG2", "NQO1", "HSD17B1"],
}


def pathway_file_is_populated(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        table = pd.read_csv(path, sep="\t")
    except Exception:
        return False
    return {"pathway_name", "gene_symbol"}.issubset(table.columns) and len(table) > 0


if not pathway_file_is_populated(PATHWAY_TSV):
    pd.DataFrame([
        {"pathway_name": pathway, "gene_symbol": gene, "gene_namespace": "HGNC symbol",
         "included": True,
         "curation_note": "Reproduced from the exploratory analysis; verify provenance before release."}
        for pathway, genes in FALLBACK_PATHWAYS.items() for gene in genes
    ]).to_csv(PATHWAY_TSV, sep="\t", index=False)
    print("Created the pathway definition file:", PATHWAY_TSV.name)

pathway_table = pd.read_csv(PATHWAY_TSV, sep="\t")
if "included" in pathway_table.columns:
    keep = pathway_table["included"].astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})
    pathway_table = pathway_table.loc[keep].copy()
pathway_table["pathway_name"] = pathway_table["pathway_name"].astype(str).str.strip()
pathway_table["gene_symbol"] = pathway_table["gene_symbol"].astype(str).str.upper().str.strip()
pathway_table = pathway_table.drop_duplicates(["pathway_name", "gene_symbol"])

pathways = pathway_table.groupby("pathway_name")["gene_symbol"].apply(list).to_dict()
with PATHWAY_GMT.open("w", encoding="utf-8") as handle:
    for pathway, genes in pathways.items():
        handle.write(f"{pathway}\tcurated_asthma_pathway\t" + "\t".join(genes) + "\n")

unique_curated_genes = sorted(set(pathway_table["gene_symbol"]))
multi_pathway = (pathway_table.groupby("gene_symbol")["pathway_name"].nunique()
                 .loc[lambda s: s > 1].sort_values(ascending=False))

print("Pathways                :", len(pathways))
print("Pathway-gene memberships:", len(pathway_table))
print("Unique curated genes    :", len(unique_curated_genes))
print("Genes in >1 pathway     :", len(multi_pathway))
print("\nThat overlap is why section 19 applies one BH correction across all unique")
print("curated genes rather than one per pathway: a gene in two pathways would otherwise")
print("receive two adjusted p-values and could be significant in one and not the other.")
print(multi_pathway.to_string())
assert len(pathways) == 10

### 9c. Probe annotation

GPL13158 and GPL570 share probe set identifiers, differing by a `_PM_` infix. Gene symbols
are therefore taken from the GPL570 annotation after stripping that infix. This is a
cross-platform identifier mapping and is described as such wherever the annotation is
used.

Probes are classified into three groups rather than silently filtered: unambiguous
(one gene symbol), ambiguous (several symbols), and unmapped. Only unambiguous probes
enter the gene-level products. The other two groups are written to audit files.

In [ ]:
annotation_path = first_existing([
    DATA_ANNOTATION / "GPL570_annotation_subset.tsv.gz",
    DATA_ANNOTATION / "GPL570_annotation_subset.tsv",
    DATA_ANNOTATION / "GPL570.txt",
    DATA_ANNOTATION / "GPL570.annot",
    SOURCE_ROOT / "GPL570.txt",
    SOURCE_ROOT / "GPL570.annot",
])

if annotation_path is None:
    import GEOparse
    print("Downloading the GPL570 annotation via GEOparse...")
    GEOparse.get_GEO("GPL570", destdir=str(DATA_ANNOTATION), silent=True)
    annotation_path = first_existing([DATA_ANNOTATION / "GPL570.txt"])
    if annotation_path is None:
        raise FileNotFoundError("GEOparse did not produce data/annotation/GPL570.txt")

print("Annotation source:", annotation_path)


def clean_probe_id(value) -> str:
    """Normalise a GPL13158 probe id to its GPL570 equivalent.
    GPL13158 uses '1007_PM_s_at' where GPL570 uses '1007_s_at'."""
    value = str(value).strip().strip('"').replace("_PM_", "_").replace("_AFFX_", "AFFX-")
    if value.startswith("PM_"):
        value = value[3:]
    return value.replace(" ", "")


def load_annotation(path):
    name = path.name.lower()
    if name.endswith((".soft", ".soft.gz", ".annot")) or name == "gpl570.txt":
        import GEOparse
        table = GEOparse.get_GEO(filepath=str(path), silent=True).table.copy()
    else:
        compression = "gzip" if path.suffix.lower() == ".gz" else "infer"
        table = pd.read_csv(path, sep="\t", compression=compression, low_memory=False)

    id_column = next((c for c in ["probe_id", "Probe_ID", "ID", "ID_REF"] if c in table.columns), None)
    symbol_column = next((c for c in ["gene_symbol_raw", "Gene_Symbol", "Gene Symbol",
                                      "Gene symbol", "GENE_SYMBOL"] if c in table.columns), None)
    entrez_column = next((c for c in ["Entrez_ID", "ENTREZ_GENE_ID", "ENTREZID",
                                      "Gene ID"] if c in table.columns), None)
    if id_column is None or symbol_column is None:
        raise ValueError(f"Could not find probe/symbol columns. Available: {table.columns.tolist()[:25]}")

    selected = pd.DataFrame({
        "probe_id_clean": table[id_column].astype(str).map(clean_probe_id),
        "gene_symbol_raw": table[symbol_column],
        "entrez_id": table[entrez_column] if entrez_column else np.nan,
    }).drop_duplicates("probe_id_clean")

    symbol = (selected["gene_symbol_raw"].astype("string").str.strip()
              .replace({"": pd.NA, "---": pd.NA, "nan": pd.NA, "NA": pd.NA}))
    ambiguous = symbol.notna() & symbol.str.contains(r"\s*///\s*|\s+//\s+", regex=True, na=False)

    selected["mapping_status"] = "unmapped"
    selected.loc[symbol.notna() & ~ambiguous, "mapping_status"] = "unambiguous"
    selected.loc[ambiguous, "mapping_status"] = "ambiguous_multi_gene"
    selected["gene_symbol"] = symbol.where(~ambiguous).str.upper()
    return selected


annotation = load_annotation(annotation_path)
probe_annotation = (
    pd.DataFrame({"probe_id": expression.index.astype(str),
                  "probe_id_clean": [clean_probe_id(v) for v in expression.index]})
    .merge(annotation, on="probe_id_clean", how="left")
)
probe_annotation["mapping_status"] = probe_annotation["mapping_status"].fillna("unmapped")

status_counts = probe_annotation["mapping_status"].value_counts()
print("\nProbe mapping outcome")
print(status_counts.to_string())
print(f"\nJoin rate: "
      f"{100 * (1 - status_counts.get('unmapped', 0) / len(probe_annotation)):.1f}% "
      "of cohort probes received a gene symbol.")

probe_annotation.to_csv(DATA_ANNOTATION / "GPL570_annotation_subset.tsv.gz",
                        sep="\t", index=False, compression="gzip")
for status, filename in [("ambiguous_multi_gene", "ambiguous_probe_mappings.tsv"),
                         ("unmapped", "unmapped_probes.tsv")]:
    probe_annotation.loc[probe_annotation["mapping_status"] == status].to_csv(
        DATA_ANNOTATION / filename, sep="\t", index=False)

PROBE_TO_GENE = probe_annotation.set_index("probe_id")["gene_symbol"]

### 9d. Statistical engine

One implementation of every test used in the notebook, so probe, gene, pathway and curated
analyses cannot diverge in their statistics.

`adjusted_binary_ols` fits `value ~ effect + covariate` for every row simultaneously via
the normal equations, which is several orders of magnitude faster than looping over 54,715
`statsmodels` fits and gives identical coefficients, standard errors and p-values.
`tests/test_engine.py` verifies that equivalence against `statsmodels.OLS` and checks the
t-tests against `scipy`.

In [ ]:
# =============================================================================
# STATISTICAL ENGINE  (cell tag: "engine")
# -----------------------------------------------------------------------------
# One source of truth for every statistical test used in this analysis.
#
#   Every statistical test used in this analysis lives here, so the probe,
#   gene, pathway and curated-gene results cannot diverge in their statistics.
#
# tests/test_engine.py extracts this cell and tests it against scipy and
# statsmodels, so the tested code and the executed code cannot drift apart.
# =============================================================================


def pooled_cohens_d(group1, group2, ddof=1):
    """Cohen's d from a pooled standard deviation, rows = features.

    ddof=1  sample variance, the standard definition
    ddof=0  population variance, for comparability with pipelines that use
            sqrt((var(a) + var(b)) / 2) as the denominator
    """
    group1 = np.asarray(group1, dtype=float)
    group2 = np.asarray(group2, dtype=float)
    n1, n2 = group1.shape[1], group2.shape[1]
    var1 = np.nanvar(group1, axis=1, ddof=ddof)
    var2 = np.nanvar(group2, axis=1, ddof=ddof)

    if ddof == 0:
        pooled_sd = np.sqrt((var1 + var2) / 2.0)
    else:
        pooled_sd = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / max(n1 + n2 - 2, 1))

    mean_difference = np.nanmean(group1, axis=1) - np.nanmean(group2, axis=1)
    return np.divide(
        mean_difference,
        pooled_sd,
        out=np.zeros_like(mean_difference),
        where=pooled_sd > 0,
    )


def benjamini_hochberg(p_values):
    """BH FDR. Non-finite p-values are set to 1.0 so they cannot gain
    significance, and they still consume a test in the correction."""
    p_values = np.asarray(p_values, dtype=float)
    clean = np.where(np.isfinite(p_values), p_values, 1.0)
    return multipletests(clean, method="fdr_bh")[1]


def two_group_test(
    matrix,
    metadata,
    *,
    group_column,
    positive_level,
    reference_level,
    equal_var,
    cohens_d_ddof=1,
):
    """Vectorised unadjusted two-sample t-test, one test per row.

    equal_var=True   Student's t, assumes equal group variances
    equal_var=False  Welch's t, does not assume equal variances
    """
    positive_samples = metadata.index[metadata[group_column] == positive_level]
    reference_samples = metadata.index[metadata[group_column] == reference_level]
    if len(positive_samples) == 0 or len(reference_samples) == 0:
        raise ValueError(
            f"Missing a comparison level in '{group_column}': "
            f"{positive_level} versus {reference_level}"
        )

    group1 = matrix.loc[:, positive_samples].to_numpy(dtype=float)
    group2 = matrix.loc[:, reference_samples].to_numpy(dtype=float)

    test = stats.ttest_ind(group1, group2, axis=1, equal_var=equal_var, nan_policy="omit")

    positive_mean = np.nanmean(group1, axis=1)
    reference_mean = np.nanmean(group2, axis=1)

    results = pd.DataFrame(
        {
            "feature_id": matrix.index.astype(str),
            "positive_level": positive_level,
            "reference_level": reference_level,
            "positive_mean": positive_mean,
            "reference_mean": reference_mean,
            # Difference of means on an already-log2 scale IS the log2 fold
            # change. No second logarithm is taken anywhere in this function.
            "mean_difference": positive_mean - reference_mean,
            "t_statistic": np.asarray(test.statistic, dtype=float),
            "p_value": np.asarray(test.pvalue, dtype=float),
            "cohens_d": pooled_cohens_d(group1, group2, ddof=cohens_d_ddof),
            "test": "Student t (equal variance)" if equal_var else "Welch t (unequal variance)",
        }
    )
    results["fdr"] = benjamini_hochberg(results["p_value"])
    return results


def adjusted_binary_ols(
    matrix,
    metadata,
    *,
    effect_column,
    positive_level,
    reference_level,
    covariate_column,
    covariate_positive_level,
    cohens_d_ddof=1,
):
    """Vectorised OLS, one model per row: value ~ effect + covariate.

    The reported coefficient is (positive_level - reference_level) holding the
    covariate fixed. This is the primary genome-wide model.
    """
    samples = metadata.index.tolist()
    y = matrix.loc[:, samples].T.to_numpy(dtype=float)
    if np.isnan(y).any():
        raise ValueError("Adjusted OLS requires a complete matrix with no missing values.")

    effect_series = metadata.loc[samples, effect_column]
    positive_mask = (effect_series == positive_level).to_numpy()
    reference_mask = (effect_series == reference_level).to_numpy()
    if positive_mask.sum() == 0 or reference_mask.sum() == 0:
        raise ValueError(
            f"Missing a comparison level in '{effect_column}': "
            f"{positive_level} versus {reference_level}"
        )

    effect = positive_mask.astype(float)
    covariate = (
        (metadata.loc[samples, covariate_column] == covariate_positive_level)
        .astype(float)
        .to_numpy()
    )

    design = np.column_stack([np.ones(len(samples)), effect, covariate])
    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError(
            "The design matrix is rank-deficient: the effect and the covariate "
            "are collinear in this sample set, so the adjusted model is not "
            "identifiable."
        )

    xtx_inverse = np.linalg.inv(design.T @ design)
    beta = xtx_inverse @ design.T @ y
    residuals = y - design @ beta
    degrees_of_freedom = design.shape[0] - design.shape[1]
    residual_variance = np.sum(residuals ** 2, axis=0) / degrees_of_freedom

    effect_beta = beta[1, :]
    effect_se = np.sqrt(xtx_inverse[1, 1] * residual_variance)
    t_statistic = np.divide(
        effect_beta, effect_se, out=np.zeros_like(effect_beta), where=effect_se > 0
    )
    p_value = 2 * stats.t.sf(np.abs(t_statistic), df=degrees_of_freedom)

    group1 = matrix.loc[:, metadata.index[positive_mask]].to_numpy(dtype=float)
    group2 = matrix.loc[:, metadata.index[reference_mask]].to_numpy(dtype=float)
    positive_mean = np.mean(group1, axis=1)
    reference_mean = np.mean(group2, axis=1)

    results = pd.DataFrame(
        {
            "feature_id": matrix.index.astype(str),
            "positive_level": positive_level,
            "reference_level": reference_level,
            "covariate": covariate_column,
            "positive_mean": positive_mean,
            "reference_mean": reference_mean,
            "unadjusted_mean_difference": positive_mean - reference_mean,
            # The adjusted effect. On an already-log2 scale this is the
            # gender-adjusted log2 fold change.
            "mean_difference": effect_beta,
            "standard_error": effect_se,
            "t_statistic": t_statistic,
            "p_value": p_value,
            "cohens_d": pooled_cohens_d(group1, group2, ddof=cohens_d_ddof),
            "df_residual": degrees_of_freedom,
            "test": f"OLS adjusted for {covariate_column}",
        }
    )
    results["fdr"] = benjamini_hochberg(results["p_value"])
    return results


def classify_results(results, fdr_threshold, log2fc_threshold):
    """Attach the two significance definitions used throughout.

    meets_fdr             FDR < threshold only
    meets_dual_threshold  FDR < threshold AND |log2 fold change| > threshold

    The dual-threshold set is a subset of the FDR set. Only the dual-threshold
    count is decomposed by direction.
    """
    out = results.copy()
    out["meets_fdr"] = out["fdr"] < fdr_threshold
    out["meets_dual_threshold"] = out["meets_fdr"] & (
        out["mean_difference"].abs() > log2fc_threshold
    )
    out["direction"] = "Not dual-threshold significant"
    out.loc[
        out["meets_dual_threshold"] & (out["mean_difference"] > 0), "direction"
    ] = f"Higher in {results['positive_level'].iloc[0]}"
    out.loc[
        out["meets_dual_threshold"] & (out["mean_difference"] < 0), "direction"
    ] = f"Higher in {results['reference_level'].iloc[0]}"
    return out


def count_summary(results, analysis_name, fdr_threshold, log2fc_threshold):
    """Return every count, separately and reconciled."""
    dual = results["meets_dual_threshold"]
    positive = int((dual & (results["mean_difference"] > 0)).sum())
    reference = int((dual & (results["mean_difference"] < 0)).sum())
    dual_total = int(dual.sum())
    fdr_only = int(results["meets_fdr"].sum())

    if dual_total != positive + reference:
        raise AssertionError(
            "Directional counts do not sum to the dual-threshold total: "
            f"{positive} + {reference} != {dual_total}"
        )

    return pd.Series(
        {
            "analysis": analysis_name,
            "test": results["test"].iloc[0],
            "positive_level": results["positive_level"].iloc[0],
            "reference_level": results["reference_level"].iloc[0],
            "features_tested": int(len(results)),
            "fdr_only": fdr_only,
            "dual_threshold": dual_total,
            "higher_in_positive": positive,
            "higher_in_reference": reference,
            "fdr_threshold": fdr_threshold,
            "abs_log2fc_threshold": log2fc_threshold,
        }
    )

## 10. Differential expression: ancestry

For each probe, log2 expression is regressed on ancestry with sex as a covariate:

```
expression ~ ancestry + sex
```

The reported coefficient is the African minus European difference with sex held constant.
Because the matrix is already on a log2 scale, that coefficient is the sex-adjusted log2
fold change. P-values are corrected by Benjamini-Hochberg across all 54,715 probes.

An unadjusted Welch's t-test is computed as a sensitivity analysis, so the influence of
the adjustment can be seen directly.

Two counts are reported and kept distinct throughout:

- **FDR-only**: probes meeting FDR < 0.05
- **Dual-threshold**: probes meeting FDR < 0.05 *and* |log2FC| > 0.5

The dual-threshold set is a subset of the FDR set. Only the dual-threshold count is
decomposed into directions, and the notebook asserts that the directional counts sum to
it.

In [ ]:
primary_summaries = []
ancestry_welch = None

if True:
    primary_ancestry = classify_results(
        adjusted_binary_ols(expression, metadata, effect_column="ancestry",
                            positive_level="African", reference_level="European",
                            covariate_column="gender", covariate_positive_level="Male"),
        CONFIG["fdr_threshold"], CONFIG["absolute_log2fc_threshold"],
    ).rename(columns={"feature_id": "probe_id"})

    ancestry_welch = classify_results(
        two_group_test(expression, metadata, group_column="ancestry",
                       positive_level="African", reference_level="European",
                       equal_var=False),
        CONFIG["fdr_threshold"], CONFIG["absolute_log2fc_threshold"],
    ).rename(columns={"feature_id": "probe_id"})

    primary_summaries = [
        count_summary(primary_ancestry, "PRIMARY ancestry (OLS adjusted for gender)",
                      CONFIG["fdr_threshold"], CONFIG["absolute_log2fc_threshold"]),
        count_summary(ancestry_welch, "SENSITIVITY ancestry (Welch, unadjusted)",
                      CONFIG["fdr_threshold"], CONFIG["absolute_log2fc_threshold"]),
    ]
    print(pd.DataFrame(primary_summaries).to_string(index=False))

    primary_ancestry.to_csv(S1_DIR / "MS1_S1A_primary_ancestry_adjusted_OLS.tsv.gz",
                            sep="\t", index=False, compression="gzip")
    ancestry_welch.to_csv(S1_DIR / "MS1_S1B_sensitivity_ancestry_welch.tsv.gz",
                          sep="\t", index=False, compression="gzip")

    print()
    print("Numbers for the revised abstract and Results:")
    s = primary_summaries[0]
    print(f"  Probes tested                 : {int(s['features_tested']):,}")
    print(f"  FDR < 0.05                    : {int(s['fdr_only']):,}")
    print(f"  FDR < 0.05 and |log2FC| > 0.5 : {int(s['dual_threshold']):,}")
    print(f"    higher in African           : {int(s['higher_in_positive']):,}")
    print(f"    higher in European          : {int(s['higher_in_reference']):,}")

## 11. Volcano and heatmap: ancestry

An annotated volcano plot and a z-scored clustermap of the top differentially expressed
probes. Gene symbols come from the audited probe annotation, so probes without an
unambiguous symbol are labelled by probe identifier rather than being silently relabelled.

In [ ]:
def annotated_volcano(results, *, title, positive_label, reference_label,
                      outpath, n_labels, fdr_threshold, log2fc_threshold):
    """Annotated volcano, matching the layout used in the exploratory notebook."""
    frame = results.copy()
    frame["gene_symbol"] = PROBE_TO_GENE.reindex(frame["probe_id"]).to_numpy()
    frame["minus_log10_fdr"] = -np.log10(frame["fdr"].clip(lower=1e-300))

    up = frame["meets_dual_threshold"] & (frame["mean_difference"] > 0)
    down = frame["meets_dual_threshold"] & (frame["mean_difference"] < 0)
    ns = ~frame["meets_dual_threshold"]

    figure, axis = plt.subplots(figsize=(14, 10))
    axis.scatter(frame.loc[ns, "mean_difference"], frame.loc[ns, "minus_log10_fdr"],
                 c=COLOUR_NS, s=8, alpha=0.4, label=f"Not significant (n={int(ns.sum()):,})")
    axis.scatter(frame.loc[down, "mean_difference"], frame.loc[down, "minus_log10_fdr"],
                 c=COLOUR_DOWN, s=16, alpha=0.75,
                 label=f"Higher in {reference_label} (n={int(down.sum()):,})")
    axis.scatter(frame.loc[up, "mean_difference"], frame.loc[up, "minus_log10_fdr"],
                 c=COLOUR_UP, s=16, alpha=0.75,
                 label=f"Higher in {positive_label} (n={int(up.sum()):,})")

    axis.axhline(-np.log10(fdr_threshold), color="black", linestyle="--", linewidth=0.9)
    for x in (-log2fc_threshold, log2fc_threshold):
        axis.axvline(x, color="black", linestyle="--", linewidth=0.9)
    axis.axvline(0, color="grey", linewidth=0.6)

    labelled = (frame.loc[frame["meets_dual_threshold"] & frame["gene_symbol"].notna()]
                .assign(rank_score=lambda d: d["minus_log10_fdr"] * d["mean_difference"].abs())
                .sort_values("rank_score", ascending=False)
                .drop_duplicates("gene_symbol")
                .head(n_labels))
    for _, row in labelled.iterrows():
        axis.annotate(row["gene_symbol"], (row["mean_difference"], row["minus_log10_fdr"]),
                      fontsize=9, fontweight="bold", xytext=(4, 4), textcoords="offset points")

    axis.set_title(title, fontsize=16, fontweight="bold")
    axis.set_xlabel(f"Log$_2$ fold change ({positive_label} - {reference_label})", fontsize=12)
    axis.set_ylabel("-log$_{10}$ FDR", fontsize=12)
    axis.legend(fontsize=10, loc="upper left")
    figure.tight_layout()
    save_figure(figure, outpath)
    plt.show()
    return labelled[["probe_id", "gene_symbol", "mean_difference", "fdr"]]


if True:
    top_ancestry_labels = annotated_volcano(
        primary_ancestry,
        title="Volcano Plot: African vs European\nDifferential Gene Expression "
              "(gender-adjusted, annotated)",
        positive_label="African", reference_label="European",
        outpath=MAIN_FIGURES / "06_volcano_plot_ancestry_annotated.png",
        n_labels=CONFIG["n_volcano_labels"],
        fdr_threshold=CONFIG["fdr_threshold"],
        log2fc_threshold=CONFIG["absolute_log2fc_threshold"])
    print("\nTop annotated genes:")
    print(top_ancestry_labels.to_string(index=False))

In [ ]:
def top_de_heatmap(results, *, title, outpath, n_top, group_column, palette):
    """Z-scored clustermap of the top differentially expressed probes."""
    frame = results.copy()
    frame["gene_symbol"] = PROBE_TO_GENE.reindex(frame["probe_id"]).to_numpy()
    top = (frame.loc[frame["meets_dual_threshold"]]
           .sort_values("fdr")
           .drop_duplicates("gene_symbol")
           .head(n_top))
    if len(top) < 2:
        print("Fewer than two probes met the threshold; heatmap skipped.")
        return None

    values = expression.loc[top["probe_id"]]
    values.index = [g if isinstance(g, str) else p
                    for g, p in zip(top["gene_symbol"], top["probe_id"])]

    column_colours = pd.DataFrame({
        "Ancestry": metadata["ancestry"].map(PALETTE_ANCESTRY),
        "Gender": metadata["gender"].map(PALETTE_GENDER),
    }, index=metadata.index)

    grid = sns.clustermap(values, z_score=0, cmap="RdBu_r", center=0,
                          col_colors=column_colours, figsize=(13, 12),
                          xticklabels=True, yticklabels=True,
                          cbar_kws={"label": "Row z-score"})
    grid.ax_heatmap.set_xticklabels(grid.ax_heatmap.get_xticklabels(), fontsize=7, rotation=90)
    grid.ax_heatmap.set_yticklabels(grid.ax_heatmap.get_yticklabels(), fontsize=7)
    grid.fig.suptitle(title, y=1.01, fontsize=14, fontweight="bold")
    save_figure(grid.fig, outpath)
    plt.show()
    return top


if True:
    top_ancestry_heatmap = top_de_heatmap(
        primary_ancestry,
        title=f"Top {CONFIG['n_top_de_genes']} DE Genes: African vs European "
              "(gender-adjusted)",
        outpath=MAIN_FIGURES / "07_heatmap_top_de_ancestry.png",
        n_top=CONFIG["n_top_de_genes"], group_column="ancestry", palette=PALETTE_ANCESTRY)
    if top_ancestry_heatmap is not None:
        top_ancestry_heatmap.to_csv(MAIN_TABLES / "MS1_Table_2_top_DE_ancestry.tsv",
                                    sep="\t", index=False)

## 12. Differential expression: sex

The reciprocal model, `expression ~ sex + ancestry`, giving the male minus female
difference with ancestry held constant. Running both contrasts through the same
specification and the same expression matrix keeps the two sets of counts comparable.

In [ ]:
primary_gender = classify_results(
    adjusted_binary_ols(expression, metadata, effect_column="gender",
                        positive_level="Male", reference_level="Female",
                        covariate_column="ancestry", covariate_positive_level="African"),
    CONFIG["fdr_threshold"], CONFIG["absolute_log2fc_threshold"],
).rename(columns={"feature_id": "probe_id"})

gender_counts = count_summary(primary_gender, "Sex (OLS adjusted for ancestry)",
                              CONFIG["fdr_threshold"], CONFIG["absolute_log2fc_threshold"])
primary_summaries.append(gender_counts)
primary_gender.to_csv(S1_DIR / "MS1_S1C_primary_sex_adjusted_OLS.tsv.gz",
                      sep="\t", index=False, compression="gzip")

print(pd.DataFrame([gender_counts]).to_string(index=False))
print()
print(f"Probes tested                 : {int(gender_counts['features_tested']):,}")
print(f"FDR < 0.05                    : {int(gender_counts['fdr_only']):,}")
print(f"FDR < 0.05 and |log2FC| > 0.5 : {int(gender_counts['dual_threshold']):,}")
print(f"  higher in male              : {int(gender_counts['higher_in_positive']):,}")
print(f"  higher in female            : {int(gender_counts['higher_in_reference']):,}")

## 13. Volcano and heatmap: sex

The sex contrast is expected to be dominated by genes on the X and Y chromosomes: *XIST*
and *TSIX* higher in females, and *RPS4Y1*, *DDX3Y*, *KDM5D*, *UTY*, *USP9Y* and *EIF1AY*
higher in males. Recovering these is a useful positive control, since it shows the pipeline
detects a large, unambiguous biological signal with the expected direction and magnitude.

In [ ]:
if True:
    top_gender_labels = annotated_volcano(
        primary_gender,
        title="Volcano Plot: Male vs Female\nDifferential Gene Expression "
              "(ancestry-adjusted, annotated)",
        positive_label="Male", reference_label="Female",
        outpath=MAIN_FIGURES / "08_volcano_plot_sex_annotated.png",
        n_labels=CONFIG["n_volcano_labels"],
        fdr_threshold=CONFIG["fdr_threshold"],
        log2fc_threshold=CONFIG["absolute_log2fc_threshold"])
    print("\nTop annotated genes:")
    print(top_gender_labels.to_string(index=False))
    print()
    print("Sex-chromosome genes are expected to dominate this contrast. Their recovery")
    print("is a positive control on the pipeline.")

In [ ]:
if True:
    top_gender_heatmap = top_de_heatmap(
        primary_gender,
        title=f"Top {CONFIG['n_top_de_genes']} DE Genes: Male vs Female (ancestry-adjusted)",
        outpath=MAIN_FIGURES / "09_heatmap_top_de_sex.png",
        n_top=CONFIG["n_top_de_genes"], group_column="gender", palette=PALETTE_GENDER)
    if top_gender_heatmap is not None:
        top_gender_heatmap.to_csv(MAIN_TABLES / "MS1_top_DE_sex.tsv", sep="\t", index=False)

## 14. Side-by-side heatmaps

The two contrasts placed beside each other. Note that they are not comparable in kind: the
sex contrast is driven by genes on the sex chromosomes, whereas the ancestry contrast is
autosomal. The panels are shown together for completeness, not as a like-for-like
comparison of effect size.

In [ ]:
if top_ancestry_heatmap is not None and top_gender_heatmap is not None:
    figure, axes = plt.subplots(1, 2, figsize=(26, 14))

    for axis, top, contrast_column, palette, title in [
        (axes[0], top_ancestry_heatmap, "ancestry", PALETTE_ANCESTRY,
         f"Top {CONFIG['n_top_de_genes']} DE Genes (African vs European)"),
        (axes[1], top_gender_heatmap, "gender", PALETTE_GENDER,
         f"Top {CONFIG['n_top_de_genes']} DE Genes (Male vs Female)"),
    ]:
        order = metadata.sort_values([contrast_column]).index
        values = expression.loc[top["probe_id"], order]
        values.index = [g if isinstance(g, str) else p
                        for g, p in zip(top["gene_symbol"], top["probe_id"])]
        zscored = values.sub(values.mean(axis=1), axis=0).div(
            values.std(axis=1).replace(0, np.nan), axis=0).fillna(0)

        sns.heatmap(zscored, cmap="RdBu_r", center=0, ax=axis,
                    xticklabels=True, yticklabels=True,
                    cbar_kws={"label": "Row z-score", "shrink": 0.5})
        axis.set_title(title, fontsize=14, fontweight="bold")
        axis.tick_params(axis="x", rotation=90, labelsize=7)
        axis.tick_params(axis="y", labelsize=7)
        for tick, sample in zip(axis.get_xticklabels(), order):
            tick.set_color(palette[metadata.loc[sample, contrast_column]])

    figure.suptitle("Side-by-side comparison: ancestry versus gender contrasts",
                    fontsize=16, fontweight="bold", y=1.01)
    figure.tight_layout()
    save_figure(figure, MAIN_FIGURES / "10_side_by_side_heatmaps.png")
    plt.show()
    print("Sample labels are coloured by group. The two panels are not comparable in kind:")
    print("the sex contrast reflects a chromosomal difference, the ancestry contrast an")
    print("autosomal expression difference.")

### 14b. Gene-level products

Three distinct products, kept separate because they answer different questions and carry
different denominators:

- **probe level** (54,715 rows): the primary result, one row per probe set
- **representative probe** (one row per gene): the probe with the smallest FDR for each
  gene, used for labelling figures and for gene-level listings
- **mean-collapsed gene matrix**: unambiguous probes averaged per gene, used as input to
  pathway scoring, enrichment and the networks

A single gene is often represented by several probe sets on this array, so the three give
different counts for the same underlying result. Any figure quoted from this notebook
should state which denominator it uses.

In [ ]:
ancestry_annotated = primary_ancestry.merge(probe_annotation, on="probe_id", how="left")
ancestry_annotated.to_csv(S1_DIR / "MS1_S1D_ancestry_probe_level_annotated.tsv.gz",
                          sep="\t", index=False, compression="gzip")

representative_probe = (
    ancestry_annotated.loc[ancestry_annotated["mapping_status"] == "unambiguous"]
    .sort_values(["gene_symbol", "fdr", "p_value", "probe_id"])
    .drop_duplicates("gene_symbol", keep="first").reset_index(drop=True)
)
representative_probe.to_csv(S1_DIR / "MS1_S1E_representative_probe_per_gene.tsv.gz",
                            sep="\t", index=False, compression="gzip")

expression_with_gene = expression.copy()
expression_with_gene["gene_symbol"] = PROBE_TO_GENE.reindex(expression.index)
expression_with_gene["mapping_status"] = (
    probe_annotation.set_index("probe_id")["mapping_status"].reindex(expression.index))
gene_expression = (expression_with_gene
                   .loc[expression_with_gene["mapping_status"] == "unambiguous"]
                   .drop(columns="mapping_status")
                   .groupby("gene_symbol")[metadata.index.tolist()].mean())
gene_expression.index.name = "gene_symbol"
gene_expression.to_csv(DATA_PROCESSED / "GSE69683_24sample_gene_expression.csv.gz",
                       compression="gzip")

print("Probe-level rows          :", len(ancestry_annotated))
print("Representative-probe genes:", len(representative_probe))
print("Mean-collapsed genes      :", gene_expression.shape[0])
print(f"\nRepresentative-probe genes meeting the dual threshold: "
      f"{int(representative_probe['meets_dual_threshold'].sum()):,}")
print()
print("Three denominators are in play. Any count taken from this notebook should state")
print("which one it uses: probe level, representative probe, or collapsed gene.")

## 15. Gene set enrichment analysis

Pre-ranked GSEA over the curated pathways, ranking genes by the sex-adjusted t statistic
so the enrichment reflects the same model as the differential expression.

Parameters are fixed and reported: minimum set size 5, maximum 500, 1,000 permutations,
seed 42. GSEA p-values are permutation-derived and therefore stochastic; the seed and
permutation count are recorded so the result can be repeated.

Three views follow: the results table, a lollipop plot ordered by normalised enrichment
score, and a significance barplot with a bubble plot of direction and magnitude.

In [ ]:
gsea_results = gene_ancestry = None
try:
    import gseapy as gp

    gene_ancestry = adjusted_binary_ols(
        gene_expression, metadata, effect_column="ancestry",
        positive_level="African", reference_level="European",
        covariate_column="gender", covariate_positive_level="Male",
    ).rename(columns={"feature_id": "gene_symbol"})
    gene_ancestry["significant"] = gene_ancestry["fdr"] < CONFIG["fdr_threshold"]
    gene_ancestry.to_csv(S1_DIR / "MS1_S1F_gene_level_ancestry_adjusted_OLS.tsv.gz",
                         sep="\t", index=False, compression="gzip")

    ranking = (gene_ancestry[["gene_symbol", "t_statistic"]].dropna()
               .drop_duplicates("gene_symbol").sort_values("t_statistic", ascending=False))

    outdir = S2_DIR / "MS1_GSEA_ancestry_adjusted"
    outdir.mkdir(parents=True, exist_ok=True)
    prerank = gp.prerank(rnk=ranking, gene_sets=pathways, outdir=str(outdir),
                         min_size=CONFIG["gsea_min_size"], max_size=CONFIG["gsea_max_size"],
                         permutation_num=CONFIG["gsea_permutations"], seed=RANDOM_SEED,
                         verbose=False, no_plot=True)
    gsea_results = prerank.res2d.copy()
    for column in ["NES", "ES", "FDR q-val", "NOM p-val"]:
        if column in gsea_results.columns:
            gsea_results[column] = pd.to_numeric(gsea_results[column], errors="coerce")
    gsea_results.to_csv(S2_DIR / "MS1_S2A_GSEA_ancestry_adjusted.tsv", sep="\t", index=False)

    print(f"Pre-ranked GSEA: min_size={CONFIG['gsea_min_size']}, "
          f"{CONFIG['gsea_permutations']} permutations, seed={RANDOM_SEED}")
    print(gsea_results.head(10).to_string(index=False))
except ImportError:
    print("GSEApy not installed. Set INSTALL_PACKAGES = True and rerun from the top.")
except Exception as exception:
    print("GSEA did not complete:", repr(exception))

In [ ]:
if gsea_results is not None and "NES" in gsea_results.columns:
    frame = gsea_results.dropna(subset=["NES"]).sort_values("NES").copy()
    frame["significant"] = frame["FDR q-val"] < CONFIG["fdr_threshold"]
    y = np.arange(len(frame))

    figure, axis = plt.subplots(figsize=(10, max(6, 0.5 * len(frame))))
    axis.hlines(y, 0, frame["NES"], color="grey", linewidth=1.2, zorder=1)
    axis.scatter(frame["NES"], y, s=110, zorder=3, edgecolors="black", linewidth=0.6,
                 c=frame["significant"].map({True: "#2c7fb8", False: "#a6bddb"}))
    axis.axvline(0, color="black", linewidth=0.8)
    axis.set_yticks(y)
    axis.set_yticklabels(frame["Term"], fontsize=10)
    axis.set_xlabel("Normalised enrichment score (NES)", fontsize=12)
    axis.set_title("GSEA: curated asthma pathways\nAfrican vs European "
                   "(gender-adjusted ranking)", fontsize=14, fontweight="bold")

    handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c,
                      markeredgecolor="black", markersize=10, label=l)
               for c, l in [("#2c7fb8", f"FDR < {CONFIG['fdr_threshold']}"),
                            ("#a6bddb", "Not significant")]]
    axis.legend(handles=handles, fontsize=10, loc="lower right")
    figure.tight_layout()
    save_figure(figure, MAIN_FIGURES / "11_gsea_lollipop_ancestry.png")
    plt.show()

In [ ]:
if gsea_results is not None and "NES" in gsea_results.columns:
    frame = gsea_results.dropna(subset=["NES"]).copy()
    frame["minus_log10_fdr"] = -np.log10(frame["FDR q-val"].clip(lower=1e-10))
    frame = frame.sort_values("minus_log10_fdr", ascending=False)

    figure, axes = plt.subplots(1, 2, figsize=(19, 6))

    colours = [COLOUR_UP if v > 0 else COLOUR_DOWN for v in frame["NES"]]
    axes[0].barh(frame["Term"], frame["minus_log10_fdr"], color=colours, alpha=0.85,
                 edgecolor="black", linewidth=0.5)
    axes[0].axvline(-np.log10(CONFIG["fdr_threshold"]), color="black",
                    linestyle="--", linewidth=1)
    axes[0].invert_yaxis()
    axes[0].set_xlabel("-log$_{10}$ FDR", fontsize=12)
    axes[0].set_title("GSEA significance by pathway", fontsize=13, fontweight="bold")
    axes[0].tick_params(labelsize=9)

    scatter = axes[1].scatter(frame["NES"], frame["Term"],
                              s=60 + 260 * frame["minus_log10_fdr"] / max(frame["minus_log10_fdr"].max(), 1e-9),
                              c=frame["NES"], cmap="RdBu_r",
                              edgecolors="black", linewidth=0.6, vmin=-2.2, vmax=2.2)
    axes[1].axvline(0, color="black", linewidth=0.8)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("Normalised enrichment score", fontsize=12)
    axes[1].set_title("Enrichment direction and magnitude\n(point size = -log$_{10}$ FDR)",
                      fontsize=13, fontweight="bold")
    axes[1].tick_params(labelsize=9)
    figure.colorbar(scatter, ax=axes[1], label="NES")

    figure.tight_layout()
    save_figure(figure, MAIN_FIGURES / "12_gsea_significance_ancestry.png")
    plt.show()

    print("Negative NES means the pathway is enriched among genes with LOWER expression in")
    print("African ancestry relative to European. State the direction explicitly in the")
    print("Results; 'enriched' alone is ambiguous.")

## 16. ssGSEA pathway scoring

Single-sample GSEA gives each sample a score per pathway, which allows pathway activity to
be modelled with the same specification used elsewhere.

Scores are tested by sex-adjusted OLS. A Wilcoxon rank-sum test is also computed for
comparison, since rank-based testing is common for ssGSEA scores. With 12 samples per group
the Wilcoxon test has limited power and coarsely discretised p-values, and it cannot
incorporate a covariate, so the adjusted model is reported as primary.

In [ ]:
pathway_scores = pathway_primary = pathway_wilcoxon = None
try:
    import gseapy as gp

    ss = gp.ssgsea(data=gene_expression, gene_sets=pathways, outdir=None,
                   sample_norm_method="rank", min_size=CONFIG["gsea_min_size"],
                   max_size=CONFIG["gsea_max_size"], permutation_num=0,
                   no_plot=True, seed=RANDOM_SEED, verbose=False)

    table = ss.res2d.copy()
    value_column = next((c for c in ["NES", "ES"] if c in table.columns), None)
    pathway_scores = table.pivot(index="Term", columns="Name", values=value_column)
    pathway_scores = pathway_scores.loc[:, metadata.index].astype(float)
    pathway_scores.index.name = "pathway"
    pathway_scores.to_csv(S2_DIR / "MS1_S2B_ssGSEA_scores.tsv", sep="\t")

    pathway_primary = adjusted_binary_ols(
        pathway_scores, metadata, effect_column="ancestry",
        positive_level="African", reference_level="European",
        covariate_column="gender", covariate_positive_level="Male",
    ).rename(columns={"feature_id": "pathway"})
    pathway_primary["significant"] = pathway_primary["fdr"] < CONFIG["fdr_threshold"]
    pathway_primary.to_csv(S2_DIR / "MS1_S2C_ssGSEA_ancestry_adjusted_OLS.tsv",
                           sep="\t", index=False)

    african = metadata.index[metadata["ancestry"] == "African"]
    european = metadata.index[metadata["ancestry"] == "European"]
    rows = []
    for pathway in pathway_scores.index:
        statistic, p_value = stats.ranksums(pathway_scores.loc[pathway, african],
                                            pathway_scores.loc[pathway, european])
        rows.append({"pathway": pathway,
                     "median_african": float(pathway_scores.loc[pathway, african].median()),
                     "median_european": float(pathway_scores.loc[pathway, european].median()),
                     "wilcoxon_statistic": float(statistic), "p_value": float(p_value)})
    pathway_wilcoxon = pd.DataFrame(rows)
    pathway_wilcoxon["fdr"] = benjamini_hochberg(pathway_wilcoxon["p_value"])
    pathway_wilcoxon["significant"] = pathway_wilcoxon["fdr"] < CONFIG["fdr_threshold"]
    pathway_wilcoxon.to_csv(S2_DIR / "MS1_S2D_ssGSEA_ancestry_wilcoxon.tsv",
                            sep="\t", index=False)

    print("PRIMARY: ssGSEA scores, sex-adjusted OLS")
    print(pathway_primary[["pathway", "mean_difference", "p_value", "fdr", "significant"]]
          .sort_values("fdr").round(4).to_string(index=False))
    print("\nCOMPARISON: ssGSEA scores, Wilcoxon rank-sum (unadjusted, rank-based)")
    print(pathway_wilcoxon.sort_values("fdr").round(4).to_string(index=False))
except ImportError:
    print("GSEApy not installed. Set INSTALL_PACKAGES = True and rerun from the top.")
except Exception as exception:
    print("ssGSEA did not complete:", repr(exception))

In [ ]:
if pathway_scores is not None:
    long_scores = (pathway_scores.T.join(metadata[["ancestry", "gender"]])
                   .melt(id_vars=["ancestry", "gender"], var_name="pathway",
                         value_name="score"))

    figure, axes = plt.subplots(1, 2, figsize=(22, 7))
    for axis, hue, palette, title in [
        (axes[0], "ancestry", PALETTE_ANCESTRY, "ssGSEA pathway scores by ancestry"),
        (axes[1], "gender", PALETTE_GENDER, "ssGSEA pathway scores by gender"),
    ]:
        sns.boxplot(data=long_scores, x="pathway", y="score", hue=hue,
                    palette=palette, ax=axis, showfliers=False, width=0.7)
        sns.stripplot(data=long_scores, x="pathway", y="score", hue=hue,
                      palette=palette, ax=axis, dodge=True, size=3,
                      edgecolor="black", linewidth=0.4, legend=False)
        axis.set_title(title, fontsize=13, fontweight="bold")
        axis.set_xlabel("")
        axis.set_ylabel("ssGSEA score")
        axis.tick_params(axis="x", rotation=90, labelsize=9)
        axis.legend(title=hue.title(), fontsize=9)

    figure.tight_layout()
    save_figure(figure, MAIN_FIGURES / "13_ssgsea_pathway_scores.png")
    plt.show()

## 17. Pathway volcano plot

Note the x-axis: these are **ssGSEA score differences**, not log2 fold changes. ssGSEA
scores are rank-based enrichment statistics, so a difference between groups is not a fold
change and must not be labelled as one.

In [ ]:
if pathway_primary is not None:
    frame = pathway_primary.copy()
    frame["minus_log10_fdr"] = -np.log10(frame["fdr"].clip(lower=1e-300))

    figure, axis = plt.subplots(figsize=(12, 8))
    axis.scatter(frame["mean_difference"], frame["minus_log10_fdr"],
                 c=np.where(frame["significant"], COLOUR_UP, COLOUR_NS),
                 s=170, edgecolors="black", linewidth=0.7, zorder=3)
    axis.axhline(-np.log10(CONFIG["fdr_threshold"]), color="black",
                 linestyle="--", linewidth=1)
    axis.axvline(0, color="black", linewidth=0.8)
    for _, row in frame.iterrows():
        axis.annotate(row["pathway"], (row["mean_difference"], row["minus_log10_fdr"]),
                      fontsize=9, xytext=(6, 5), textcoords="offset points")
    axis.set_title("Volcano Plot: Pathway Differences (African vs European)\n"
                   "gender-adjusted ssGSEA scores", fontsize=16, fontweight="bold")
    axis.set_xlabel("ssGSEA score difference (African - European, gender-adjusted)", fontsize=12)
    axis.set_ylabel("-log$_{10}$ FDR", fontsize=12)
    figure.tight_layout()
    save_figure(figure, MAIN_FIGURES / "14_volcano_ssgsea_pathways.png")
    plt.show()

## 18. Violin plots for curated genes by pathway

One figure per pathway, as in the original notebook: expression of the curated genes split
by ancestry.

In [ ]:
pathway_violin_paths = []

for pathway_name, genes in pathways.items():
    present = [g for g in genes if g in gene_expression.index]
    if len(present) < 2:
        print(f"Skipping {pathway_name}: fewer than two genes present on the array.")
        continue

    melted = (gene_expression.loc[present].T
              .join(metadata[["ancestry", "gender"]])
              .melt(id_vars=["ancestry", "gender"], var_name="gene", value_name="expression"))

    figure, axis = plt.subplots(figsize=(max(8, 0.85 * len(present)), 6))
    sns.violinplot(data=melted, x="gene", y="expression", hue="ancestry",
                   palette=PALETTE_ANCESTRY, split=True, inner="quartile",
                   cut=0, ax=axis, linewidth=1)
    sns.stripplot(data=melted, x="gene", y="expression", hue="ancestry",
                  palette=PALETTE_ANCESTRY, dodge=True, size=2.5, ax=axis,
                  edgecolor="black", linewidth=0.3, legend=False)
    axis.set_title(f"{pathway_name.replace('_', ' ')}: curated gene expression by ancestry",
                   fontsize=13, fontweight="bold")
    axis.set_xlabel("")
    axis.set_ylabel("Log$_2$ expression")
    axis.tick_params(axis="x", rotation=90, labelsize=9)
    axis.legend(title="Ancestry", fontsize=9)

    outpath = FIG_PATHWAY / f"violin_{pathway_name}.png"
    figure.tight_layout()
    save_figure(figure, outpath)
    pathway_violin_paths.append(outpath)
    plt.show()

print(f"\n{len(pathway_violin_paths)} pathway violin figures written to "
      f"{FIG_PATHWAY.name}/")

## 19. Curated gene analysis

Benjamini-Hochberg correction is applied **once across all unique curated genes**, rather
than separately within each pathway. This matters because 11 of the curated genes belong
to more than one pathway: correcting within pathways would give those genes several
adjusted p-values and allow the same gene to be significant in one pathway and not
another. A single family across all unique genes gives each gene one answer.

Both contrasts are computed so the ancestry and sex effects on the curated set can be
compared side by side.

In [ ]:
curated_present = [g for g in unique_curated_genes if g in gene_expression.index]
curated_expression = gene_expression.loc[curated_present]

print(f"Curated genes defined : {len(unique_curated_genes)}")
print(f"Curated genes present : {len(curated_present)}")
missing_curated = sorted(set(unique_curated_genes) - set(curated_present))
if missing_curated:
    print(f"Not on the array or unmapped ({len(missing_curated)}): {missing_curated}")

# PRIMARY: one correction family across all curated genes.
curated_primary = adjusted_binary_ols(
    curated_expression, metadata, effect_column="ancestry",
    positive_level="African", reference_level="European",
    covariate_column="gender", covariate_positive_level="Male",
).rename(columns={"feature_id": "gene_symbol"})
curated_primary["significant"] = curated_primary["fdr"] < CONFIG["fdr_threshold"]
curated_primary["correction_family"] = f"BH across all {len(curated_present)} curated genes"

# Gender contrast for the curated genes, for the paired volcano below.
curated_gender = adjusted_binary_ols(
    curated_expression, metadata, effect_column="gender",
    positive_level="Male", reference_level="Female",
    covariate_column="ancestry", covariate_positive_level="African",
).rename(columns={"feature_id": "gene_symbol"})
curated_gender["significant"] = curated_gender["fdr"] < CONFIG["fdr_threshold"]

# COMPARISON: BH applied within each pathway instead of once across all genes.
membership = pathway_table[["pathway_name", "gene_symbol"]].drop_duplicates()
per_pathway_rows = []
for pathway_name, group in membership.groupby("pathway_name"):
    genes = [g for g in group["gene_symbol"] if g in gene_expression.index]
    if len(genes) < 2:
        continue
    result = two_group_test(gene_expression.loc[genes], metadata, group_column="ancestry",
                            positive_level="African", reference_level="European",
                            equal_var=True, cohens_d_ddof=0).rename(
                                columns={"feature_id": "gene_symbol"})
    result["fdr"] = benjamini_hochberg(result["p_value"])   # per-pathway family
    result["pathway_name"] = pathway_name
    per_pathway_rows.append(result)

curated_per_pathway = pd.concat(per_pathway_rows, ignore_index=True)
curated_per_pathway["significant"] = curated_per_pathway["fdr"] < CONFIG["fdr_threshold"]

per_pathway_significant = sorted(set(curated_per_pathway.loc[curated_per_pathway["significant"], "gene_symbol"]))
primary_significant = sorted(curated_primary.loc[curated_primary["significant"], "gene_symbol"])

print(f"\nSignificant under per-pathway correction: {len(per_pathway_significant)}")
print(f"  {per_pathway_significant}")
print(f"\nSignificant under ONE correction family (PRIMARY): {len(primary_significant)}")
print(f"  {primary_significant}")
print(f"\nSignificant per-pathway only : {sorted(set(per_pathway_significant) - set(primary_significant))}")
print(f"Significant single-family only: {sorted(set(primary_significant) - set(per_pathway_significant))}")
print("\nThe per-pathway family is more permissive because each correction is applied")
print("over a smaller set of tests. The single family is reported.")

curated_primary.to_csv(S3_DIR / "MS1_S3A_curated_genes_PRIMARY_single_family.tsv",
                       sep="\t", index=False)
curated_per_pathway.to_csv(S3_DIR / "MS1_S3B_curated_genes_per_pathway_comparison.tsv",
                      sep="\t", index=False)
curated_gender.to_csv(S3_DIR / "MS1_S3C_curated_genes_gender_adjusted.tsv",
                      sep="\t", index=False)
(curated_primary.loc[curated_primary["significant"]].sort_values("fdr")
 .to_csv(MAIN_TABLES / "MS1_Table_3_curated_genes_revised.tsv", sep="\t", index=False))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(22, 8))

for axis, frame, positive, reference, title in [
    (axes[0], curated_primary, "African", "European",
     "Curated genes: African vs European\n(gender-adjusted)"),
    (axes[1], curated_gender, "Male", "Female",
     "Curated genes: Male vs Female\n(ancestry-adjusted)"),
]:
    plot_frame = frame.copy()
    plot_frame["minus_log10_fdr"] = -np.log10(plot_frame["fdr"].clip(lower=1e-300))
    plot_frame["group"] = "Not significant"
    plot_frame.loc[plot_frame["significant"] & (plot_frame["mean_difference"] > 0),
                   "group"] = f"Higher in {positive}"
    plot_frame.loc[plot_frame["significant"] & (plot_frame["mean_difference"] < 0),
                   "group"] = f"Higher in {reference}"

    for label, colour in [("Not significant", COLOUR_NS),
                          (f"Higher in {reference}", COLOUR_DOWN),
                          (f"Higher in {positive}", COLOUR_UP)]:
        subset = plot_frame["group"] == label
        axis.scatter(plot_frame.loc[subset, "mean_difference"],
                     plot_frame.loc[subset, "minus_log10_fdr"],
                     color=colour, s=70, alpha=0.85, edgecolors="black", linewidth=0.4,
                     label=f"{label} (n={int(subset.sum())})")

    axis.axhline(-np.log10(CONFIG["fdr_threshold"]), color="black",
                 linestyle="--", linewidth=0.9)
    axis.axvline(0, color="black", linewidth=0.8)
    for _, row in plot_frame.loc[plot_frame["significant"]].iterrows():
        axis.annotate(row["gene_symbol"], (row["mean_difference"], row["minus_log10_fdr"]),
                      fontsize=8, fontweight="bold", xytext=(4, 4), textcoords="offset points")
    axis.set_title(title, fontsize=14, fontweight="bold")
    axis.set_xlabel(f"Adjusted log$_2$ fold change ({positive} - {reference})", fontsize=11)
    axis.set_ylabel("-log$_{10}$ FDR", fontsize=11)
    axis.legend(title="Upregulated in", fontsize=9, title_fontsize=10)

figure.tight_layout()
save_figure(figure, MAIN_FIGURES / "15_volcano_curated_genes.png")
plt.show()

In [ ]:
# Violin plot of the significant curated genes, as in the original notebook.
significant_genes = curated_primary.loc[curated_primary["significant"], "gene_symbol"].tolist()

if significant_genes:
    melted = (gene_expression.loc[significant_genes].T
              .join(metadata[["ancestry", "gender"]])
              .melt(id_vars=["ancestry", "gender"], var_name="gene", value_name="expression"))

    figure, axis = plt.subplots(figsize=(max(10, 0.95 * len(significant_genes)), 6))
    sns.violinplot(data=melted, x="gene", y="expression", hue="ancestry",
                   palette=PALETTE_ANCESTRY, split=True, inner="quartile",
                   cut=0, ax=axis, linewidth=1)
    sns.stripplot(data=melted, x="gene", y="expression", hue="ancestry",
                  palette=PALETTE_ANCESTRY, dodge=True, size=3, ax=axis,
                  edgecolor="black", linewidth=0.3, legend=False)
    axis.set_title(f"Significantly different curated genes by ancestry "
                   f"(n={len(significant_genes)}, FDR < {CONFIG['fdr_threshold']}, "
                   "gender-adjusted)", fontsize=14, fontweight="bold")
    axis.set_xlabel("")
    axis.set_ylabel("Log$_2$ expression")
    axis.tick_params(axis="x", rotation=90, labelsize=10)
    axis.legend(title="Ancestry", fontsize=10)
    figure.tight_layout()
    save_figure(figure, MAIN_FIGURES / "16_violin_significant_curated_genes.png")
    plt.show()

## 20. Co-expression networks

Spearman co-expression among the curated genes, with a single pre-specified threshold
(|r| >= 0.60 and FDR < 0.05) applied identically to every subgroup.

Subgroup networks are built on 8 to 12 samples, where correlation estimates are highly
unstable. They are exploratory. Edge counts should not be compared between subgroups of
different size as though the difference were biological: a network on 8 samples and one on
16 are not on a comparable footing.

In [ ]:
PATHWAY_MEMBERSHIP = (pathway_table.groupby("gene_symbol")["pathway_name"]
                      .apply(lambda names: sorted(set(names))[0]).to_dict())
pathway_colour_map = dict(zip(sorted(pathways), sns.color_palette("tab10", len(pathways))))


def build_coexpression_network(matrix, sample_ids_subset, group_name, output_directory):
    subset = matrix.loc[:, list(sample_ids_subset)]
    subset = subset.loc[subset.var(axis=1, ddof=1) > 0]
    genes = subset.index.tolist()

    correlations, p_values = stats.spearmanr(subset.to_numpy(dtype=float), axis=1)
    if np.ndim(correlations) == 0:
        correlations = np.array([[1.0, correlations], [correlations, 1.0]])
        p_values = np.array([[0.0, p_values], [p_values, 0.0]])

    iu = np.triu_indices(len(genes), k=1)
    pairs = pd.DataFrame({
        "gene1": np.asarray(genes)[iu[0]], "gene2": np.asarray(genes)[iu[1]],
        "spearman_r": np.nan_to_num(correlations[iu], nan=0.0),
        "p_value": np.nan_to_num(p_values[iu], nan=1.0),
    })
    pairs["fdr"] = benjamini_hochberg(pairs["p_value"])
    edges = pairs.loc[(pairs["fdr"] < CONFIG["coexpression_fdr"])
                      & (pairs["spearman_r"].abs() >= CONFIG["coexpression_abs_r"])].copy()

    graph = nx.Graph()
    graph.add_nodes_from(genes)
    for _, edge in edges.iterrows():
        graph.add_edge(edge["gene1"], edge["gene2"],
                       weight=float(edge["spearman_r"]), fdr=float(edge["fdr"]))

    degree = dict(graph.degree())
    betweenness = nx.betweenness_centrality(graph) if graph.number_of_nodes() > 1 else {}
    nodes = pd.DataFrame({
        "gene": genes,
        "pathway": [PATHWAY_MEMBERSHIP.get(g, "Unassigned") for g in genes],
        "degree": [degree.get(g, 0) for g in genes],
        "betweenness": [betweenness.get(g, 0.0) for g in genes],
    }).sort_values("degree", ascending=False)

    slug = re.sub(r"[^A-Za-z0-9]+", "_", group_name).strip("_").lower()
    pairs.to_csv(output_directory / f"MS1_S4_{slug}_all_pairs.tsv.gz",
                 sep="\t", index=False, compression="gzip")
    edges.to_csv(output_directory / f"MS1_S4_{slug}_edges.tsv", sep="\t", index=False)
    nodes.to_csv(output_directory / f"MS1_S4_{slug}_nodes.tsv", sep="\t", index=False)

    connected = [g for g in graph.nodes if degree.get(g, 0) > 0]
    figure, axis = plt.subplots(figsize=(11, 10))
    if not connected:
        axis.text(0.5, 0.5,
                  f"No edges met |r| >= {CONFIG['coexpression_abs_r']} and "
                  f"FDR < {CONFIG['coexpression_fdr']}\n{group_name} "
                  f"(n={len(sample_ids_subset)} samples)",
                  ha="center", va="center", fontsize=12)
    else:
        sub = graph.subgraph(connected)
        positions = nx.spring_layout(sub, seed=RANDOM_SEED, k=0.65)
        node_colours = [pathway_colour_map.get(PATHWAY_MEMBERSHIP.get(g, ""), "#999999")
                        for g in sub.nodes]
        nx.draw_networkx_nodes(sub, positions, ax=axis, node_color=node_colours,
                               node_size=[110 + 45 * degree.get(n, 0) for n in sub.nodes],
                               alpha=0.9, edgecolors="black", linewidths=0.5)
        nx.draw_networkx_edges(sub, positions, ax=axis, alpha=0.55,
                               width=[abs(d["weight"]) * 2 for *_, d in sub.edges(data=True)])
        nx.draw_networkx_labels(sub, positions, ax=axis, font_size=8)
        present_pathways = sorted({PATHWAY_MEMBERSHIP.get(g, "Unassigned") for g in sub.nodes})
        handles = [Line2D([0], [0], marker="o", color="w", markersize=9,
                          markerfacecolor=pathway_colour_map.get(p, "#999999"),
                          markeredgecolor="black", label=p.replace("_", " "))
                   for p in present_pathways]
        axis.legend(handles=handles, title="Pathway", bbox_to_anchor=(1.02, 1),
                    loc="upper left", fontsize=8, title_fontsize=9)

    axis.set_title(f"Co-expression network: {group_name}\n"
                   f"|Spearman r| >= {CONFIG['coexpression_abs_r']}, "
                   f"FDR < {CONFIG['coexpression_fdr']} (pre-specified)",
                   fontsize=13, fontweight="bold")
    axis.axis("off")
    figure.tight_layout()
    save_figure(figure, output_directory / f"MS1_S4_{slug}_network.png")
    plt.show()

    return {"group": group_name, "samples": len(sample_ids_subset), "genes": len(genes),
            "tested_pairs": len(pairs), "retained_edges": len(edges),
            "connected_nodes": len(connected),
            "abs_r_threshold": CONFIG["coexpression_abs_r"],
            "fdr_threshold": CONFIG["coexpression_fdr"]}

In [ ]:
network_summary = pd.DataFrame()

if RUN_NETWORKS:
    network_genes = sorted(g for g in unique_curated_genes if g in gene_expression.index)
    network_expression = gene_expression.loc[network_genes]
    print(f"Network genes: {len(network_genes)}")

    groups = {
        "All samples":       metadata.index,
        "African ancestry":  metadata.index[metadata["ancestry"] == "African"],
        "European ancestry": metadata.index[metadata["ancestry"] == "European"],
        "Female":            metadata.index[metadata["gender"] == "Female"],
        "Male":              metadata.index[metadata["gender"] == "Male"],
    }

    summaries = []
    for name, ids in groups.items():
        if len(ids) < 6:
            print(f"Skipping {name}: only {len(ids)} samples.")
            continue
        summaries.append(build_coexpression_network(network_expression, ids, name, S4_DIR))

    network_summary = pd.DataFrame(summaries)
    network_summary.to_csv(S4_DIR / "MS1_S4_network_summary.tsv", sep="\t", index=False)
    print(network_summary.to_string(index=False))
    print()
    print("No threshold was adjusted after inspecting any subgroup. Subgroup networks use")
    print("8 to 12 samples, where Spearman correlations are highly unstable. Describe them")
    print("as exploratory and do not compare edge counts between subgroups of different")
    print("size as though the difference were biological.")

## 21. Summary of findings

The summary below is assembled from values computed in this run, so the written text
cannot drift from the numbers it describes.

In [ ]:
lines = []
add = lines.append

add("CROSS-ANCESTRY ASTHMA TRANSCRIPTOMICS: SUMMARY OF FINDINGS")
add("=" * 74)
add(f"Generated {RUN_STARTED}")
add(f"Dataset {CONFIG['dataset']}, platform {CONFIG['platform']}, "
    f"{expression.shape[0]:,} probes, {expression.shape[1]} samples")
add("")

add("COHORT")
add("-" * 74)
add(f"  {int((metadata['ancestry'] == 'African').sum())} African ancestry, "
    f"{int((metadata['ancestry'] == 'European').sum())} European ancestry")
add(f"  {int((metadata['gender'] == 'Female').sum())} female, "
    f"{int((metadata['gender'] == 'Male').sum())} male; 8 female and 4 male per ancestry group")
add("  Ancestry is self-reported race or ethnicity as recorded by U-BIOPRED.")
add("  No genotype data are available, so this is not genetically inferred ancestry.")
add("")

if True:
    s = primary_summaries[0]
    add("PRIMARY ANCESTRY ANALYSIS (OLS adjusted for gender)")
    add("-" * 74)
    add(f"  Probes tested                 : {int(s['features_tested']):,}")
    add(f"  FDR < 0.05                    : {int(s['fdr_only']):,}")
    add(f"  FDR < 0.05 and |log2FC| > 0.5 : {int(s['dual_threshold']):,}")
    add(f"    higher in African           : {int(s['higher_in_positive']):,}")
    add(f"    higher in European          : {int(s['higher_in_reference']):,}")
    add("  Report the two counts as separate quantities. The directional counts belong")
    add("  to the dual-threshold subset, not to the FDR-only count.")
    add("")

add(f"CURATED ASTHMA GENES ({len(curated_present)} of {len(unique_curated_genes)} present)")
add("-" * 74)
add(f"  Significant, single correction family: {len(primary_significant)}")
add(f"    {', '.join(primary_significant) if primary_significant else 'none'}")
add(f"  A per-pathway correction would give {len(per_pathway_significant)}; the single")
add("  family across all curated genes is reported.")
add("")

if pathway_primary is not None:
    significant_pathways = pathway_primary.loc[pathway_primary["significant"]].sort_values("fdr")
    add("PATHWAY-LEVEL DIFFERENCES (ssGSEA, gender-adjusted)")
    add("-" * 74)
    if len(significant_pathways):
        for _, row in significant_pathways.iterrows():
            direction = "higher in African" if row["mean_difference"] > 0 else "higher in European"
            add(f"  {row['pathway']:<32} score difference {row['mean_difference']:+.4f} "
                f"({direction}), FDR {row['fdr']:.4f}")
    else:
        add("  No pathway met FDR < 0.05 after adjustment.")
    add("")

if gsea_results is not None and "NES" in gsea_results.columns:
    enriched = gsea_results.dropna(subset=["NES"])
    enriched = enriched.loc[enriched["FDR q-val"] < CONFIG["fdr_threshold"]].sort_values("FDR q-val")
    add("PRE-RANKED GSEA")
    add("-" * 74)
    if len(enriched):
        for _, row in enriched.iterrows():
            direction = "African" if row["NES"] > 0 else "European"
            add(f"  {row['Term']:<32} NES {row['NES']:+.3f} (towards {direction}), "
                f"FDR {row['FDR q-val']:.4f}")
    else:
        add("  No gene set met FDR < 0.05.")
    add("")

add("INTERPRETATION AND LIMITS")
add("-" * 74)
add("  Whole blood is a cell mixture. Ancestry-associated differences in leucocyte")
add("  composition, particularly the Duffy-null neutrophil phenotype, are a plausible")
add("  driver of part of this signal. Deconvolution is not performed here, so")
add("  cell-composition and cell-intrinsic effects cannot be separated.")
add("")
add("  With 12 samples per ancestry group the study is underpowered for small effects.")
add("  Subgroup co-expression networks use 8 to 12 samples and are exploratory only.")
add("")
add("  Quality control identified no outlier samples. It does not establish the absence")
add("  of batch effects; no batch covariate is available in the GEO metadata.")
add("")
add("  Findings are associations within a single severe-asthma cohort and require")
add("  replication in an independent, ancestry-diverse dataset before any clinical")
add("  or mechanistic claim is made.")

summary_text = "\n".join(lines)
(MAIN_TABLES / "MS1_findings_summary.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)

## 22. Validation, manifest and checksums

Design and internal-consistency checks, then a SHA-256 index of every generated file and a
record of the software environment.

In [ ]:
checks = []


def check(name, passed, detail=""):
    checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})


check("Sample count matches design", expression.shape[1] == CONFIG["expected_samples"],
      expression.shape[1])
check("Expression columns aligned to metadata",
      list(expression.columns) == list(metadata.index), "exact order and membership")
check("No missing expression values", not expression.isna().any().any(), 0)
check("Ancestry composition matches design",
      metadata["ancestry"].value_counts().to_dict() == CONFIG["expected_ancestry_counts"],
      metadata["ancestry"].value_counts().to_dict())
check("Sex composition matches design",
      metadata["gender"].value_counts().to_dict() == CONFIG["expected_sex_counts"],
      metadata["gender"].value_counts().to_dict())
check("Probe count matches the expected platform",
      expression.shape[0] == CONFIG["expected_probes"], expression.shape[0])
check("Ten curated pathways loaded", len(pathways) == 10, len(pathways))
check("Ancestry and sex are not collinear",
      pd.crosstab(metadata["ancestry"], metadata["gender"]).min().min() > 0,
      "the adjusted models are identifiable")
check("Each ancestry group maps to one GEO race value",
      int(cohort_race.groupby("manifest_label")["geo_race_value"].nunique().max()) == 1,
      "no categories pooled")
check("Log2FC is a difference of means, not a ratio of logs",
      np.allclose(primary_ancestry["unadjusted_mean_difference"],
                  primary_ancestry["positive_mean"] - primary_ancestry["reference_mean"]),
      "verified elementwise")
check("Directional counts sum to the dual-threshold total",
      int(primary_ancestry["meets_dual_threshold"].sum())
      == int((primary_ancestry["meets_dual_threshold"] & (primary_ancestry["mean_difference"] > 0)).sum())
       + int((primary_ancestry["meets_dual_threshold"] & (primary_ancestry["mean_difference"] < 0)).sum()),
      "ancestry contrast")
check("FDR-only count is at least the dual-threshold count",
      int(primary_ancestry["meets_fdr"].sum()) >= int(primary_ancestry["meets_dual_threshold"].sum()),
      f"{int(primary_ancestry['meets_fdr'].sum()):,} >= "
      f"{int(primary_ancestry['meets_dual_threshold'].sum()):,}")
check("Sex contrast recovers known sex-linked genes",
      bool(PROBE_TO_GENE.reindex(
          primary_gender.loc[primary_gender["meets_dual_threshold"], "probe_id"]
      ).isin(["XIST", "RPS4Y1", "DDX3Y", "KDM5D", "UTY", "USP9Y", "EIF1AY"]).any()),
      "positive control for the pipeline")
check("Co-expression threshold fixed at the pre-specified value",
      CONFIG["coexpression_abs_r"] == 0.60, CONFIG["coexpression_abs_r"])

validation = pd.DataFrame(checks)
validation.to_csv(MAIN_TABLES / "MS1_validation_report.tsv", sep="\t", index=False)
print(validation.to_string(index=False))

failed = validation.loc[~validation["passed"]]
print()
if len(failed):
    print(f"{len(failed)} CHECK(S) FAILED: resolve before using these results.")
    print(failed[["check", "detail"]].to_string(index=False))
else:
    print("All validation checks passed.")

In [ ]:
rows = []
for section in [MAIN_TABLES, MAIN_FIGURES, S1_DIR, S2_DIR, S3_DIR, S4_DIR]:
    for file_path in sorted(section.rglob("*")):
        if file_path.is_file() and file_path.name != "MANIFEST.tsv":
            rows.append({
                "section": file_path.parent.relative_to(MS1).as_posix(),
                "relative_path": file_path.relative_to(REPO_ROOT).as_posix(),
                "size_bytes": file_path.stat().st_size,
                "sha256": sha256_file(file_path),
                "generated_utc": RUN_STARTED,
            })

manifest_frame = pd.DataFrame(rows)
manifest_frame.to_csv(MS1 / "MANIFEST.tsv", sep="\t", index=False)
print(f"Indexed {len(manifest_frame)} generated files.")
print(manifest_frame.groupby("section")["relative_path"].count().to_string())

In [ ]:
from importlib import metadata as importlib_metadata

package_versions = {}
for package in ["numpy", "pandas", "scipy", "statsmodels", "scikit-learn", "matplotlib",
                "seaborn", "GEOparse", "gseapy", "networkx", "requests"]:
    try:
        package_versions[package] = importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        package_versions[package] = "not installed"

environment = {
    "run_started_utc": RUN_STARTED,
    "python": sys.version,
    "platform": platform.platform(),
    "random_seed": RANDOM_SEED,
    "repository_root": str(REPO_ROOT),
    "config": CONFIG,
    "inputs": {
        "series_matrix": str(series_matrix_path),
        "series_matrix_sha256": SERIES_MATRIX_SHA256,
        "annotation": str(annotation_path),
        "pathway_definitions": str(PATHWAY_TSV),
        "cohort_source": manifest_source,
    },
    "options": {"sensitivity_analysis": RUN_SENSITIVITY, "networks": RUN_NETWORKS},
    "packages": package_versions,
}
(MAIN_TABLES / "MS1_environment.json").write_text(
    json.dumps(environment, indent=2, default=str), encoding="utf-8")
print(json.dumps(environment, indent=2, default=str))

## 23. Supplementary assembly

Builds the four supplementary sets from the files this run generated, each with a README
and checksums, and archives them for deposition.

In [ ]:
SUPPLEMENT_README = {
    "S1_complete_DE_and_expression":
        "Complete differential expression results at probe and gene level, PCA and t-SNE\n"
        "coordinates, and the sample correlation matrix. Primary results come from OLS\n"
        "adjusted for the other factor: sex for the ancestry contrast, ancestry for the\n"
        "sex contrast. The unadjusted Welch sensitivity analysis is included alongside.",
    "S2_pathway_and_enrichment":
        "Pre-ranked GSEA and ssGSEA results. ssGSEA scores are tested by sex-adjusted OLS\n"
        "and, for comparison, by Wilcoxon rank-sum. A single pre-specified FDR threshold\n"
        "of 0.05 is used throughout.",
    "S3_curated_gene_statistics":
        "Per-gene statistics for the curated asthma gene set. The primary file applies one\n"
        "Benjamini-Hochberg correction across all unique curated genes. A comparison file\n"
        "shows the effect of correcting within each pathway instead.",
    "S4_coexpression_networks":
        "Spearman co-expression networks for the curated genes, overall and by subgroup.\n"
        "A single pre-specified threshold (|r| >= 0.60, FDR < 0.05) is applied to every\n"
        "subgroup. Subgroup networks use 8 to 12 samples and are exploratory.",
}

for directory in [S1_DIR, S2_DIR, S3_DIR, S4_DIR]:
    name = directory.name
    files = sorted(f for f in directory.rglob("*") if f.is_file() and f.name != "README.txt")
    text = [f"{name}", "=" * len(name), "",
            SUPPLEMENT_README.get(name, ""), "",
            f"Generated {RUN_STARTED} from {CONFIG['dataset']} "
            f"(series matrix SHA-256 {SERIES_MATRIX_SHA256[:16]}...)",
            f"Random seed {RANDOM_SEED}", "",
            f"Contents ({len(files)} files):", ""]
    for f in files:
        text.append(f"  {f.relative_to(directory).as_posix():<62} "
                    f"{f.stat().st_size:>12,} bytes")
        text.append(f"      sha256 {sha256_file(f)}")
    (directory / "README.txt").write_text("\n".join(text), encoding="utf-8")
    print(f"{name}: {len(files)} files, README written")

archive_base = REPO_ROOT / "results" / f"MS1_supplementary_{RUN_STARTED[:10]}"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(SUPPLEMENT))
print("\nSupplementary archive:", Path(archive_path).name)
print("Size (MB):", round(Path(archive_path).stat().st_size / 1e6, 2))

## Reusing this notebook

**A different cohort from GSE69683.** Replace `data/manifests/sample_manifest.tsv` with
your sample list and update `expected_samples`, `expected_ancestry_counts` and
`expected_sex_counts` in `CONFIG`. The design assertions in section 5 will fail loudly if
the manifest and the configuration disagree, which is the intended behaviour.

**A different gene set.** Replace `data/gene_sets/asthma_curated_pathways.tsv`. The GMT is
regenerated from it, so the two cannot fall out of step. Provenance columns are provided
for recording where each gene came from.

**A different dataset.** The parsing in section 4 assumes a GEO series matrix. The
statistical engine takes any features-by-samples matrix and a metadata frame, so sections
10 onward are dataset-independent given those two inputs.

**Different thresholds.** Everything lives in `CONFIG`. Nothing below section 1 hard-codes
a threshold or a path.

---

### Citation

If you use this notebook, please cite the release DOI rather than the repository URL, so
the citation resolves to the exact code that produced the results. Cite the data
separately: Bigler et al. (2017), GEO accession GSE69683.